# 09 · Matrix factorizations / Factorizaciones matriciales

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/project-delphi/tensors-workshop/blob/main/notebooks/09-matrix-factorizations.ipynb)

<div style="height:3px;border-radius:2px;margin:1.4em 0 1.6em;background:linear-gradient(90deg,#c2410c,rgba(194,65,12,0))"></div>

<span style="font:700 11px/1.6 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;color:#c2410c">PART IV · EXERCISE · 15 MIN</span>

## Practise today / Practica hoy

Compare solver residuals and coefficient sensitivity; trace reduced coordinates through reconstruction.

<div style="border-left:4px solid rgba(130,130,150,.5);background:rgba(130,130,150,.09);border-radius:0 8px 8px 0;padding:12px 16px;margin:1.2em 0 1.8em;font:400 14.5px/1.7 ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"><div style="font:700 10.5px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;opacity:.62;margin-bottom:10px">🇪🇸 ESPAÑOL</div><div style="margin:0 0 0">Comparar residuos y sensibilidad de coeficientes; seguir coordenadas reducidas hasta la reconstrucción.</div></div>

## Explore later / Explora después

Benchmark other factorizations, compress images with SVD, and inspect NMF factors.

<div style="border-left:4px solid rgba(130,130,150,.5);background:rgba(130,130,150,.09);border-radius:0 8px 8px 0;padding:12px 16px;margin:1.2em 0 1.8em;font:400 14.5px/1.7 ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"><div style="font:700 10.5px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;opacity:.62;margin-bottom:10px">🇪🇸 ESPAÑOL</div><div style="margin:0 0 0">Medir otras factorizaciones, comprimir imágenes con SVD e inspeccionar factores NMF.</div></div>

Follow the core block immediately below. / Sigue el bloque esencial de abajo.

<!-- CORE-PATH -->
## Core path / Ruta esencial

Read and run this block from top to bottom: **recall → example → attempt → feedback → checkpoint**. Preparation and feedback definitions appear where needed. Try before opening a folded solution. Stop at **Core complete**; everything after it is **Explore later**.

🇪🇸 Lee y ejecuta este bloque de arriba abajo: **recuerda → ejemplo → intento → retroalimentación → comprobación**. La preparación y las funciones de comprobación aparecen donde se necesitan. Inténtalo antes de abrir una solución plegada. Detente en **Fin de la ruta esencial**; después empieza **Explora después**.

### Recall / Recuerda

In Notebook 07, could identical predictions identify individual coefficients?

🇪🇸 En el cuaderno 07, ¿predicciones idénticas identificaban los coeficientes individuales?

## Setup / Preparación

Run this cell first. Every cell below depends on it.

It loads four things the workshop already uses, so there is nothing new to download except the airline CSV you also fetched in section 08:

1. **Two real images** — `astronaut()` and `camera()` from `skimage.data`, both 512×512 in grayscale.
2. **Real handwritten digits** — `load_digits()`, 1797 images of 8×8 pixels, flattened to a 1797×64 matrix.
3. **The real monthly airline series** from section 08 — 144 months of passenger counts.
4. **A flop-count table**, used later to turn big-O into predicted seconds.

> 🇪🇸 Ejecuta primero esta celda; todas las celdas de abajo dependen de ella.
>
> Carga cuatro cosas que el taller ya usa, así que no hay nada nuevo que descargar salvo el CSV de aerolíneas que también obtuviste en la sección 08:
>
> 1. **Dos imágenes reales** — `astronaut()` y `camera()`, ambas de 512×512 en escala de grises.
> 2. **Dígitos manuscritos reales** — 1797 imágenes de 8×8 píxeles, aplanadas a una matriz de 1797×64.
> 3. **La serie real mensual de aerolíneas** de la sección 08 — 144 meses.
> 4. **Una tabla de conteo de flops**, que después convierte la notación big-O en segundos predichos.

### Core prep 1/1 · Preparación esencial

Run the next cell. / Ejecuta la siguiente celda.

In [ ]:
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import ipywidgets as widgets

from IPython.display import display
from scipy.linalg import cho_factor, cho_solve, lu_factor
from skimage import data
from skimage.color import rgb2gray
from sklearn.datasets import load_digits
from sklearn.decomposition import NMF

try:
    from google.colab import output
    output.enable_custom_widget_manager()
except ImportError:
    pass

FLIGHTS = (
    "https://raw.githubusercontent.com/mwaskom/"
    "seaborn-data/master/flights.csv"
)

# 1. Two real images, both already workshop datasets.
IMAGES = {
    "astronaut": rgb2gray(data.astronaut()),
    "camera": data.camera().astype(float) / 255.0,
}

# 2. Real handwritten digits, scaled to [0, 1]: 1797 rows, 64 pixel columns.
digits = load_digits()
D = digits.data / 16.0
digit_labels = digits.target

# 3. The real monthly airline series from section 08.
flights = pd.read_csv(FLIGHTS)
passengers = flights["passengers"].to_numpy(float)
month = np.arange(len(passengers), dtype=float)
month_scaled = month / month.max()          # rescaled to [0, 1]

rng = np.random.default_rng(0)

# 4. Flop counts for a dense m x n factorization with m >= n, from
# Trefethen & Bau, *Numerical Linear Algebra*. Only the randomized SVD
# uses the target rank k.
FLOPS = {
    "Cholesky": lambda m, n, k: n ** 3 / 3,
    "LU": lambda m, n, k: 2 * n ** 3 / 3,
    "QR": lambda m, n, k: 2 * m * n ** 2 - 2 * n ** 3 / 3,
    "Eigendecomposition": lambda m, n, k: 9 * n ** 3,
    "Thin SVD": lambda m, n, k: 2 * m * n ** 2 + 11 * n ** 3,
    "Randomized SVD": lambda m, n, k: 4 * m * n * k,
}

def relative_error(reference, approximation):
    return (np.linalg.norm(reference - approximation)
            / np.linalg.norm(reference))

def psnr(reference, approximation, peak=1.0):
    """Peak signal-to-noise ratio in dB, for images scaled to [0, peak]."""
    mse = np.mean((reference - approximation) ** 2)
    return 10 * np.log10(peak ** 2 / mse)

def best_time(call, repeats=3):
    """Fastest of `repeats` runs — the least noisy estimator of a timing."""
    times = []
    for _ in range(repeats):
        start = time.perf_counter()
        call()
        times.append(time.perf_counter() - start)
    return min(times)

print("Images / Imágenes:", {k: v.shape for k, v in IMAGES.items()})
print("Digit matrix / Matriz de dígitos:", D.shape)
print("Airline months / Meses de aerolíneas:", passengers.shape)
print("Flop formulas / Fórmulas de flops:", len(FLOPS))
print()
print("EN: Setup ready.")
print("ES: Preparación lista.")

### Example / Ejemplo

For a tall matrix `X`, least squares chooses coefficients to minimize `||X @ beta - y||`. QR writes `X = Q @ R`, with orthonormal columns in `Q`, so solve `R @ beta = Q.T @ y`. Forming `X.T @ X` instead squares the condition number for full-column-rank `X`; this can amplify numerical error. The residual compares predictions with data. A coefficient comparison asks a different question.

🇪🇸 Para una matriz alta `X`, mínimos cuadrados elige coeficientes que minimizan `||X @ beta - y||`. QR escribe `X = Q @ R`, con columnas ortonormales en `Q`, y se resuelve `R @ beta = Q.T @ y`. Formar `X.T @ X` eleva al cuadrado el número de condición si `X` tiene rango columna completo; esto puede amplificar errores numéricos. El residuo compara predicciones con datos. Comparar coeficientes responde otra pregunta.

### One matrix, three factors / Una matriz, tres factores

<div style="height:3px;border-radius:2px;margin:1.6em 0 1.9em;background:linear-gradient(90deg,#c2410c,rgba(194,65,12,0))"></div>

The SVD splits a matrix into three, and truncating it keeps only the largest directions. The rank-1 approximation is already close in the big rows and visibly wrong in the small one — which is the whole trade-off, at a size you can check by hand.

<img src="https://project-delphi.github.io/tensors-workshop/images/cube-09-svd.gif" alt="An animation of the singular value decomposition of a 4 by 3 matrix into U, a diagonal S, and V transpose, followed by rank-1 and rank-2 approximations shown beside the original matrix." style="max-width:100%;display:block;margin:2.2em auto;border-radius:10px">

<div style="border-left:4px solid rgba(130,130,150,.5);background:rgba(130,130,150,.09);border-radius:0 8px 8px 0;padding:15px 19px;margin:1.8em 0 2.2em;font:400 14.5px/1.8 ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"><div style="font:700 10.5px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;opacity:.62;margin-bottom:10px">🇪🇸 ESPAÑOL</div>La SVD divide una matriz en tres, y truncarla conserva solo las direcciones mayores. La aproximación de rango 1 ya se acerca en las filas grandes y falla a la vista en la pequeña: ese es todo el compromiso, a un tamaño que puedes comprobar a mano.</div>

## Exercise 1 — least squares two ways / Ejercicio 1 — mínimos cuadrados de dos maneras

<div style="height:3px;border-radius:2px;margin:1.6em 0 1.9em;background:linear-gradient(90deg,#c2410c,rgba(194,65,12,0))"></div>

Least squares is one problem. QR and normal equations are two routes. Compare stability, then speed.

Fit a degree-10 polynomial to 144 monthly airline counts.

`X = np.vander(month_scaled, 11, increasing=True)` has shape `(144, 11)`.

Compare `solve(X.T @ X, X.T @ y)` with QR followed by `solve(R, Q.T @ y)`. Use `np.linalg.lstsq` as the reference.

**Predict:** Can similar residuals hide different coefficients? Report both.

<div style="border-left:4px solid rgba(130,130,150,.5);background:rgba(130,130,150,.09);border-radius:0 8px 8px 0;padding:15px 19px;margin:1.8em 0 2.2em;font:400 14.5px/1.8 ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"><div style="font:700 10.5px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;opacity:.62;margin-bottom:10px">🇪🇸 ESPAÑOL</div>Ajusta un polinomio de grado 10 a 144 meses. Compara ecuaciones normales y QR con `lstsq`. ¿Pueden coincidir los residuos y diferir los coeficientes? Informa ambos.</div>

### Check your work / Comprueba tu trabajo

Run the next cell once; it defines `check_core_answer` without solving the exercise. After your attempt, pass your two coefficient vectors, three residuals (normal, QR, reference), and two relative coefficient errors (normal, QR). It checks numerical consistency; explain your solver choice yourself.

Ejecuta la siguiente celda una vez: define `check_core_answer` sin resolver el ejercicio. Tras tu intento, pasa tus dos vectores de coeficientes, tres residuos (normal, QR, referencia) y dos errores relativos de coeficientes (normal, QR). Comprueba coherencia numérica; justifica tú la elección del método.


In [ ]:
def check_core_answer(X, y, beta_normal, beta_qr, residuals, coefficient_errors):
    """Check solver outputs and reported metrics / Comprueba métodos y medidas."""
    X, y = np.asarray(X), np.asarray(y)
    assert X.ndim == 2 and y.shape == (X.shape[0],), "Check X/y shapes / Revisa las formas de X/y."
    betas = [np.asarray(beta_normal), np.asarray(beta_qr)]
    assert all(b.shape == (X.shape[1],) for b in betas), "One coefficient per column / Un coeficiente por columna."
    assert all(np.isfinite(b).all() for b in betas), "Coefficients must be finite / Los coeficientes deben ser finitos."
    reference = np.linalg.lstsq(X, y, rcond=None)[0]
    assert np.linalg.norm(reference) > 0, "Relative coefficient error needs a nonzero reference / El error relativo exige una referencia no nula."
    normal = np.linalg.solve(X.T @ X, X.T @ y)
    assert np.allclose(betas[0], normal, rtol=1e-6, atol=1e-8), "Recheck normal equations / Revisa las ecuaciones normales."
    assert np.linalg.norm(betas[1] - reference) <= 1e-4 * max(np.linalg.norm(reference), 1e-12), "Recheck Q.T @ y and solve(R, ...) / Revisa Q.T @ y y solve(R, ...)."
    expected_residuals = [np.linalg.norm(X @ b - y) for b in [*betas, reference]]
    expected_errors = [np.linalg.norm(b - reference) / np.linalg.norm(reference) for b in betas]
    assert np.shape(residuals) == (3,), "Report normal, QR, reference residuals / Informa los residuos normal, QR y referencia."
    assert np.allclose(residuals, expected_residuals, rtol=1e-6, atol=1e-8), "Residual is ||X @ beta - y|| / El residuo es ||X @ beta - y||."
    assert np.shape(coefficient_errors) == (2,), "Report two coefficient errors / Informa dos errores de coeficientes."
    assert np.allclose(coefficient_errors, expected_errors, rtol=1e-5, atol=1e-12), "Divide coefficient difference by ||beta_ref|| / Divide la diferencia entre ||beta_ref||."
    print("Checks passed; explain both metrics / Comprobaciones superadas; explica ambas medidas.")
    return True


### Core activity · Actividad esencial

**Predict → Run → Explain → Check**

1. **Predict.** Can two fits have similar residuals but different coefficients?
2. **Run.** Complete Exercise 1; open one hint at a time if stuck.
3. **Explain.** Use both measurements to explain your solver choice.
4. **Check.** Compare with lstsq and report conditioning. It is a numerical reference, not known true coefficients. Call `check_core_answer(X, y, beta_normal, beta_qr, residuals, coefficient_errors)` before opening the solution.

<details>
<summary>Español · Predice → Ejecuta → Explica → Comprueba</summary>

1. **Predice.** ¿Dos ajustes pueden tener residuos similares y coeficientes distintos?
2. **Ejecuta.** Completa el Ejercicio 1; abre una pista a la vez si te atascas.
3. **Explica.** Justifica el método con ambas medidas.
4. **Comprueba.** Compara con lstsq e informa el condicionamiento. Es una referencia numérica, no los coeficientes verdaderos. Llama a `check_core_answer(X, y, beta_normal, beta_qr, residuals, coefficient_errors)` antes de abrir la solución.

</details>

Prediction / Predicción: ___  
Evidence / Evidencia: ___  
Revised explanation / Explicación revisada: ___

<details>
<summary>Hint 1 / Pista 1 · Shapes / Formas</summary>

`X` is `(144, 11)`, `y` is `(144,)`, and each coefficient vector is `(11,)`. What shape must `Q.T @ y` have?

`X` tiene forma `(144, 11)`, `y` tiene `(144,)` y cada vector de coeficientes tiene `(11,)`. ¿Qué forma debe tener `Q.T @ y`?

</details>

<details>
<summary>Hint 2 / Pista 2 · Measures / Medidas</summary>

A residual compares predictions with observations: `np.linalg.norm(X @ beta - y)`. Coefficient error compares vectors: `np.linalg.norm(beta - beta_ref) / np.linalg.norm(beta_ref)`.

El residuo compara predicciones con observaciones: `np.linalg.norm(X @ beta - y)`. El error de coeficientes compara vectores: `np.linalg.norm(beta - beta_ref) / np.linalg.norm(beta_ref)`.

</details>

In [ ]:
# TODO 1 / TAREA 1
#
# EN:
# 1. Build X = np.vander(month_scaled, 11, increasing=True); y = passengers.
# 2. Print cond(X) and cond(X.T @ X); compare the latter with cond(X)**2.
# 3. Normal equations: beta_normal = np.linalg.solve(X.T @ X, X.T @ y).
# 4. QR: Q, R = np.linalg.qr(X); beta_qr = np.linalg.solve(R, Q.T @ y).
# 5. Reference: beta_ref = np.linalg.lstsq(X, y, rcond=None)[0].
# 6. Set residuals to [||X @ beta - y||] for normal, QR, reference, in order.
# 7. Set coefficient_errors to [||beta - beta_ref|| / ||beta_ref||]
#    for normal and QR, in order. Report both lists and explain differences.
# 8. Call check_core_answer(X, y, beta_normal, beta_qr, residuals, coefficient_errors).
#
# ES:
# 1. Construye X = np.vander(month_scaled, 11, increasing=True); y = passengers.
# 2. Imprime cond(X) y cond(X.T @ X); compara la segunda con cond(X)**2.
# 3. Ecuaciones normales: beta_normal = np.linalg.solve(X.T @ X, X.T @ y).
# 4. QR: Q, R = np.linalg.qr(X); beta_qr = np.linalg.solve(R, Q.T @ y).
# 5. Referencia: beta_ref = np.linalg.lstsq(X, y, rcond=None)[0].
# 6. Define residuals como [||X @ beta - y||] para normal, QR y referencia, en orden.
# 7. Define coefficient_errors como [||beta - beta_ref|| / ||beta_ref||]
#    para normal y QR, en orden. Informa ambas listas y explica diferencias.
# 8. Llama a check_core_answer(X, y, beta_normal, beta_qr, residuals, coefficient_errors).


In [ ]:
#@title Solution / Solución — try it yourself first / inténtalo primero { display-mode: 'form' }

X = np.vander(month_scaled, 11, increasing=True)
y = passengers

cond_X = np.linalg.cond(X)
cond_normal = np.linalg.cond(X.T @ X)

beta_normal = np.linalg.solve(X.T @ X, X.T @ y)

Q, R = np.linalg.qr(X)
beta_qr = np.linalg.solve(R, Q.T @ y)

beta_ref = np.linalg.lstsq(X, y, rcond=None)[0]

eps = np.finfo(float).eps

print("Design matrix / Matriz de diseño:", X.shape)
print("cond(X)      =", f"{cond_X:.3e}")
print("cond(XtX)    =", f"{cond_normal:.3e}",
      "   cond(X)^2 =", f"{cond_X ** 2:.3e}")
print()
print("Conditioning × machine precision (scale, not guarantee) / Condicionamiento × precisión (escala, no garantía):")
print("   QR             kappa(X)   * eps =", f"{cond_X * eps:.3e}")
print("   normal eqs.    kappa(X)^2 * eps =", f"{cond_X ** 2 * eps:.3e}")
print()

residual = lambda b: np.linalg.norm(X @ b - y)

print("Residual ||X beta - y|| / Residuo:")
print("   normal equations / ecuaciones normales:", f"{residual(beta_normal):.6f}")
print("   QR                                    :", f"{residual(beta_qr):.6f}")
print("   lstsq (reference / referencia)        :", f"{residual(beta_ref):.6f}")
print()

coef_error = lambda b: (np.linalg.norm(b - beta_ref)
                        / np.linalg.norm(beta_ref))

print("Relative coefficient error / Error relativo de coeficientes:")
print("   normal equations / ecuaciones normales:", f"{coef_error(beta_normal):.3e}")
print("   QR                                    :", f"{coef_error(beta_qr):.3e}")
print()
residuals = [residual(b) for b in (beta_normal, beta_qr, beta_ref)]
coefficient_errors = [coef_error(b) for b in (beta_normal, beta_qr)]
check_core_answer(X, y, beta_normal, beta_qr, residuals, coefficient_errors)
print("EN: compare your measured residuals and coefficient errors; the exact gap varies by numerical library.")
print("ES: compara tus residuos y errores de coeficientes; la brecha exacta depende de la librería numérica.")


### Residual check · 3 minutes / Comprobación del residuo · 3 minutos

For `A = diag([1, 1e-8])`, compare coefficients `[1, 0]` and `[1, 1]` against observations `[1, 0]`. Predict both residual norms. Can a tiny residual settle which coefficient vector to trust?

🇪🇸 Para `A = diag([1, 1e-8])`, compara coeficientes `[1, 0]` y `[1, 1]` con observaciones `[1, 0]`. Predice las dos normas residuales. ¿Un residuo pequeño basta para confiar en los coeficientes?

<details><summary>Check / Comprueba</summary>

The residual norms are 0 and `1e-8`, despite a coefficient difference of 1. The second direction is weakly constrained: report conditioning and coefficient sensitivity as well as residuals.

Las normas residuales son 0 y `1e-8`, aunque los coeficientes difieren en 1. La segunda dirección está débilmente determinada: informa condicionamiento y sensibilidad de coeficientes además de residuos.

</details>

### Checkpoint: SVD to Tucker (3 minutes) / Comprobación: de SVD a Tucker (3 minutos)

SVD writes `A = U @ diag(s) @ V.T`: paired input/output directions and their strengths. Retaining the largest strength and its two directions gives a rank-1 approximation.

SVD escribe `A = U @ diag(s) @ V.T`: direcciones emparejadas de entrada/salida y sus intensidades. Conservar la mayor intensidad y sus dos direcciones da una aproximación de rango 1.

**Predict by hand before opening the check.** Let `A = [[3, 0], [0, 1]]` and retain the first singular direction `U₁ = V₁ = [[1], [0]]`. Compute `G = U₁.T @ A @ V₁`, then `A_hat = U₁ @ G @ V₁.T`. Write both shapes and values. Does returning to shape `(2, 2)` recover every original value?

**Predice a mano antes de abrir la comprobación.** Sea `A = [[3, 0], [0, 1]]` y conserva la primera dirección singular `U₁ = V₁ = [[1], [0]]`. Calcula `G = U₁.T @ A @ V₁` y luego `A_hat = U₁ @ G @ V₁.T`. Escribe ambas formas y valores. ¿Volver a la forma `(2, 2)` recupera todos los valores originales?

Prediction / Predicción: `G = ___`, shape / forma `___`; `A_hat = ___`, shape / forma `___`.

<details>
<summary>Check and connect / Comprueba y conecta</summary>

`G = [[3]]` has shape `(1, 1)`. It contains coordinates in the retained bases, not a cropped data table. `A_hat = [[3, 0], [0, 0]]` has shape `(2, 2)`: expanding the coordinates restores the axes, but the discarded value `1` is lost. The absolute Frobenius error is `1`.

For a matrix `(I, J)`, bases `(I, r₁)` and `(J, r₂)` give core `(r₁, r₂)`. Tucker adds a third basis `(K, r₃)` and core `(r₁, r₂, r₃)`. Reconstruction has shape `(I, J, K)` even when values change. In Notebook 10, say what the three original axes and three reduced axes mean.

`G = [[3]]` tiene forma `(1, 1)`. Contiene coordenadas en las bases retenidas, no una tabla de datos recortada. `A_hat = [[3, 0], [0, 0]]` tiene forma `(2, 2)`: expandir las coordenadas recupera los ejes, pero se pierde el valor descartado `1`. El error absoluto de Frobenius es `1`.

Para una matriz `(I, J)`, bases `(I, r₁)` y `(J, r₂)` dan un núcleo `(r₁, r₂)`. Tucker añade una tercera base `(K, r₃)` y un núcleo `(r₁, r₂, r₃)`. La reconstrucción tiene forma `(I, J, K)` aunque cambien los valores. En el Notebook 10, explica qué significan los tres ejes originales y los tres reducidos.

</details>

## Core complete / Fin de la ruta esencial

Keep your prediction, evidence, and explanation. Follow the facilitator’s quiz and break schedule before continuing.

🇪🇸 Guarda tu predicción, evidencia y explicación. Sigue las pausas y quizzes del facilitador antes de continuar.

[Next: Notebook 10 / Siguiente: cuaderno 10](https://colab.research.google.com/github/project-delphi/tensors-workshop/blob/main/notebooks/10-tucker-decomposition.ipynb).

## Explore later / Explora después

Optional reference, exercises, and explorers. These are outside this section’s live core. Continue in order when studying them; some reuse earlier setup.

🇪🇸 Material de consulta, ejercicios y exploradores opcionales. Quedan fuera de la ruta esencial en vivo. Continúa en orden al estudiarlos; algunos reutilizan la preparación anterior.

The [SVD portal](https://project-delphi.github.io/tensors-workshop/interactive/linalg-stage.html?lang=en#portal) is the picture under this exercise: a unit circle on the left, the ellipse `A` maps it to on the right, and `A vᵢ = σᵢ uᵢ` reached by scrubbing rather than asserted. The ratio `σ₁/σ₂` you can see there is the condition number that just cost you your coefficients. The step after it, the [ellipsoid](https://project-delphi.github.io/tensors-workshop/interactive/linalg-stage.html?lang=en#ellipsoid), is the same picture one dimension up: `σ₁/σ₃` as the proportion of a shape, with the shortest axis on a slider.

<div style="border-left:4px solid rgba(130,130,150,.5);background:rgba(130,130,150,.09);border-radius:0 8px 8px 0;padding:15px 19px;margin:1.8em 0 2.2em;font:400 14.5px/1.8 ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"><div style="font:700 10.5px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;opacity:.62;margin-bottom:10px">🇪🇸 ESPAÑOL</div>El <a href="https://project-delphi.github.io/tensors-workshop/interactive/linalg-stage.html?lang=es#portal">portal de la SVD</a> es la imagen que hay debajo de este ejercicio: una circunferencia unidad a la izquierda, la elipse a la que <code>A</code> la lleva a la derecha, y <code>A vᵢ = σᵢ uᵢ</code> al que se llega desplazando en lugar de afirmarlo. El cociente <code>σ₁/σ₂</code> que puedes ver ahí es el número de condición que te acaba de costar los coeficientes. El paso siguiente, el <a href="https://project-delphi.github.io/tensors-workshop/interactive/linalg-stage.html?lang=es#ellipsoid">elipsoide</a>, es la misma imagen una dimensión más arriba: <code>σ₁/σ₃</code> como la proporción de una forma, con el eje más corto en un deslizador.</div>

## Predict first / Predice primero

<div style="height:3px;border-radius:2px;margin:1.6em 0 1.9em;background:linear-gradient(90deg,#c2410c,rgba(194,65,12,0))"></div>

Someone says the following. **Decide whether they are right before you
reveal anything** — commit to one answer, then open the check.

<div style="border-left:5px solid #c2410c;background:rgba(194,65,12,0.1);border-radius:0 8px 8px 0;padding:18px 22px;margin:2.2em 0;font:400 15px/1.85 ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"><div style="font:700 11px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;color:#c2410c;margin-bottom:11px">THE CLAIM · LA AFIRMACIÓN</div><div style="margin:.55em 0">&ldquo;Both fits are almost exact, so <b>their coefficients must agree</b>.&rdquo;</div></div>

A prediction you have committed to is worth more than one you keep
adjusting as the answer appears.

<div style="border-left:4px solid rgba(130,130,150,.5);background:rgba(130,130,150,.09);border-radius:0 8px 8px 0;padding:15px 19px;margin:1.8em 0 2.2em;font:400 14.5px/1.8 ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"><div style="font:700 10.5px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;opacity:.62;margin-bottom:10px">🇪🇸 ESPAÑOL</div>Alguien afirma que, como ambos ajustes son casi exactos, sus <b>coeficientes</b> deben coincidir. Decide si tiene razón <b>antes</b> de revelar la comprobación.</div>

In [ ]:
#@title 🤔 Predict: does a tiny residual pin the coefficients? / Predice: ¿un residuo diminuto fija los coeficientes? — run me / ejecútame { display-mode: 'form' }

# --- counterexample / contraejemplo (tested in tests/test_teaching_materials.py) ---
import numpy as np

pred_X = np.array([[1., 1.], [1., 1.000001]])
pred_a, pred_b = np.array([1., 1.]), np.array([2., 0.])
pred_y = pred_X @ pred_a

assert np.linalg.norm(pred_X @ pred_a - pred_y) == 0
assert np.isclose(
    np.linalg.norm(pred_X @ pred_b - pred_y), 1e-6
)
assert np.isclose(
    np.linalg.norm(pred_a - pred_b), np.sqrt(2)
)
# --- end counterexample / fin del contraejemplo ---

# --- how the question is laid out / cómo se presenta la pregunta ---
# Radio buttons rather than a dropdown. Four bilingual answers squeezed into
# one 640px line were hard to read, and a dropdown hides three of them until
# you open it -- the wrong shape for a question whose whole point is weighing
# the options against each other. One per line, with room around them.
# Botones de opción en vez de un desplegable: una respuesta por línea.
import contextlib
import html as pred_html
import io

PRED_ACCENT = "#c2410c"
PRED_SANS = "ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"
PRED_MONO = "ui-monospace,SFMono-Regular,Menlo,Consolas,monospace"


def pred_tag(text):
    """A small EN / ES marker, in words rather than in colour alone."""
    return (f'<span style="font:700 10px/1 {PRED_MONO};letter-spacing:.16em;'
            f'color:{PRED_ACCENT};opacity:.8;margin-right:9px;'
            f'vertical-align:.12em">{text}</span>')


def pred_is_measurement(line):
    """True for a printed reading, false for a sentence.

    A reading wants monospace and tight rows so the numbers line up under one
    another; a sentence wants prose type and room. The two used to share one
    13px monospace column, which is most of why the reveal read as a wall.
    """
    if ":" not in line:
        return False
    tail = line.rsplit(":", 1)[1].strip()
    return bool(tail) and (tail[0].isdigit()
                           or tail[0] in "[(-+."
                           or tail.startswith(("True", "False", "nan", "inf")))


def pred_panel(text):
    """The reveal, laid out instead of printed.

    Exactly the same words: `check_prediction` still prints, and this catches
    what it printed and gives it typography. EN and ES stay written out as
    tags rather than becoming a colour, because a reader who cannot see the
    colour still has to be able to tell the two apart.
    """
    blocks = []
    for line in text.rstrip("\n").split("\n"):
        stripped = line.strip()
        if not stripped:
            blocks.append('<div style="height:12px"></div>')
        elif stripped.startswith(("EN:", "ES:")):
            tag, body = stripped[:2], stripped[3:].strip()
            blocks.append(
                f'<p style="margin:.55em 0;font:400 15px/1.8 {PRED_SANS}">'
                f'{pred_tag(tag)}{pred_html.escape(body)}</p>')
        elif pred_is_measurement(stripped):
            blocks.append(
                f'<div style="font:400 13.5px/2.0 {PRED_MONO};'
                f'white-space:pre-wrap">{pred_html.escape(stripped)}</div>')
        else:
            blocks.append(
                f'<p style="margin:.55em 0;font:600 15.5px/1.75 {PRED_SANS}">'
                f'{pred_html.escape(stripped)}</p>')
    return (f'<div style="border-left:4px solid {PRED_ACCENT};'
            f'background:rgba(130,130,150,.08);border-radius:0 10px 10px 0;'
            f'padding:16px 20px;margin:.4em 0 0">{"".join(blocks)}</div>')


def pred_render(choice, reveal):
    """Run the check, catch what it prints, and show it laid out."""
    caught = io.StringIO()
    with contextlib.redirect_stdout(caught):
        check_prediction(choice, reveal)
    display(widgets.HTML(pred_panel(caught.getvalue())))

pred_choice = widgets.RadioButtons(
    options=[
        ("— choose one / elige una —", None),
        ("They must agree closely / Deben coincidir de cerca", "agree"),
        ("Both fits must be wrong / Ambos ajustes deben estar mal", "wrong"),
        ("They can differ enormously / Pueden diferir enormemente", "differ"),
    ],
    value=None,
    description="",
    layout=widgets.Layout(width="auto", margin="0 0 6px 0"),
)

pred_reveal = widgets.Checkbox(
    value=False,
    description="Show me the answer / Muéstrame la respuesta",
    indent=False,
    layout=widgets.Layout(margin="10px 0 4px 0"),
)

def check_prediction(choice, reveal):
    if choice is None:
        print("Choose an answer first / Elige una respuesta primero.")
        return

    if not reveal:
        print("Answer saved / Respuesta guardada.")
        print("Tick the box above when you are ready / Marca la casilla de arriba\n      cuando quieras.".replace("\n      ", " "))
        return

    print("Prediction gap / Diferencia en la predicción:",
          f"{np.linalg.norm(pred_X @ pred_b - pred_y):.7f}")
    print("Coefficient gap / Diferencia en coeficientes:",
          f"{np.linalg.norm(pred_a - pred_b):.7f}")
    print()
    print("Coefficients / Coeficientes:", pred_a, "vs", pred_b)
    print()
    if choice == "differ":
        print("You were right / Acertaste.")
    else:
        print("You were wrong — read on / Te equivocaste; sigue leyendo.")
    print()
    print("EN: predictions differ by about 0.000001 while the coefficients differ by about 1.414. Nearly dependent columns make this possible — inspect conditioning, not only fit.")
    print("ES: las predicciones difieren en unos 0,000001 mientras que los coeficientes difieren en unos 1,414. Columnas casi dependientes lo permiten: revisa el condicionamiento, no solo el ajuste.")

# The one <style> block in these notebooks, and the markdown rule does not
# cover it. ipywidgets gives no way to set the space between radio options
# from Python, and this is *widget output*, not a markdown cell: Colab strips
# <style> from markdown -- which is why every box in these notebooks is
# inline-styled -- but renders it in an output, the same path pandas' own
# Styler uses. Scoped to one added class so it can reach nothing else, and if
# it is ever dropped the options still work, just closer together.
pred_choice.add_class("pred-radio")

display(widgets.HTML(
    "<style>"
    ".pred-radio .widget-radio-box label{display:flex;align-items:flex-start;"
    "margin:0 0 13px;font:400 15px/1.6 " + PRED_SANS + "}"
    ".pred-radio input[type=radio]{flex:none;margin:4px 11px 0 0;"
    "transform:scale(1.15)}"
    "</style>"
))

pred_output = widgets.interactive_output(
    pred_render,
    {"choice": pred_choice, "reveal": pred_reveal},
)

pred_heading = widgets.HTML(
    f'<div style="font:700 11px/1.6 {PRED_MONO};letter-spacing:.18em;'
    f'color:{PRED_ACCENT};margin:2px 0 12px">'
    f'YOUR PREDICTION \u00b7 TU PREDICCI\u00d3N</div>'
)

display(widgets.VBox(
    [pred_heading, pred_choice, pred_reveal, pred_output],
    layout=widgets.Layout(padding="2px 0 14px 0"),
))

## Start with an everyday analogy / Empecemos con una analogía cotidiana

<div style="height:3px;border-radius:2px;margin:1.6em 0 1.9em;background:linear-gradient(90deg,#c2410c,rgba(194,65,12,0))"></div>

A factorization rewrites a matrix so one job becomes easy.

`3960 = 2³ × 3² × 5 × 11`. Same number. Useful pieces.

**Ask:** Which job do these factors make easier?

<div style="border-left:4px solid rgba(130,130,150,.5);background:rgba(130,130,150,.09);border-radius:0 8px 8px 0;padding:15px 19px;margin:1.8em 0 2.2em;font:400 14.5px/1.8 ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"><div style="font:700 10.5px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;opacity:.62;margin-bottom:10px">🇪🇸 ESPAÑOL</div>El mismo número, piezas útiles. ¿Qué tarea facilitan los factores?</div>

## Six factorizations, six questions / Seis factorizaciones, seis preguntas

<div style="height:3px;border-radius:2px;margin:1.6em 0 1.9em;background:linear-gradient(90deg,#c2410c,rgba(194,65,12,0))"></div>

Do not memorize six names. Match the matrix and the question to the method.

| Method / Método | Use / Uso | Requires / Requiere |
|---|---|---|
| LU | Repeated solves / Resolver muchas veces | Square, nonsingular / Cuadrada, invertible |
| Cholesky | Repeated solves / Resolver muchas veces | Symmetric positive definite / Simétrica definida positiva |
| QR | Least squares / Mínimos cuadrados | Full column rank for the simple solve below / Columnas independientes para la resolución de abajo |
| Eigen / Espectral | Repeated action / Aplicación repetida | Square; `eigh` needs symmetry / Cuadrada; `eigh` exige simetría |
| SVD | Low-rank approximation / Aproximación de rango bajo | Any matrix / Cualquier matriz |
| NMF | Additive parts / Partes aditivas | Nonnegative data / Datos no negativos |

<div style="border-left:4px solid rgba(130,130,150,.5);background:rgba(130,130,150,.09);border-radius:0 8px 8px 0;padding:15px 19px;margin:1.8em 0 2.2em;font:400 14.5px/1.8 ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"><div style="font:700 10.5px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;opacity:.62;margin-bottom:10px">🇪🇸 ESPAÑOL</div>Usa la tabla para elegir según la tarea y los requisitos.</div>

## Why this matters / Por qué esto importa

<div style="height:3px;border-radius:2px;margin:1.6em 0 1.9em;background:linear-gradient(90deg,#c2410c,rgba(194,65,12,0))"></div>

The call is short. Choosing the right call is the work.

<div style="border-left:4px solid rgba(130,130,150,.5);background:rgba(130,130,150,.09);border-radius:0 8px 8px 0;padding:15px 19px;margin:1.8em 0 2.2em;font:400 14.5px/1.8 ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"><div style="font:700 10.5px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;opacity:.62;margin-bottom:10px">🇪🇸 ESPAÑOL</div>La llamada es corta. Elegir la llamada correcta es el trabajo.</div>

## 9.1 What the factors must satisfy / Qué deben cumplir los factores

<div style="height:3px;border-radius:2px;margin:1.6em 0 1.9em;background:linear-gradient(90deg,#c2410c,rgba(194,65,12,0))"></div>

Each method trades a constraint for a useful structure. State the constraint first.

LU and Cholesky are exact rewrites: `PA = LU` and `A = LLᵀ`.

QR gives `X = QR`, with orthonormal columns in `Q` and triangular `R`.

Truncated SVD minimizes `‖A − B‖_F` with `rank(B) ≤ k`.

NMF approximates `A ≈ WH` with `W,H ≥ 0`.

<div style="border-left:4px solid rgba(130,130,150,.5);background:rgba(130,130,150,.09);border-radius:0 8px 8px 0;padding:15px 19px;margin:1.8em 0 2.2em;font:400 14.5px/1.8 ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"><div style="font:700 10.5px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;opacity:.62;margin-bottom:10px">🇪🇸 ESPAÑOL</div>LU y Cholesky reescriben exactamente la matriz. QR aporta columnas ortonormales y un factor triangular. SVD truncada minimiza el error a rango fijo. NMF exige factores no negativos.</div>

### The singular values as a budget / Los valores singulares como presupuesto

<div style="height:3px;border-radius:2px;margin:1.6em 0 1.9em;background:linear-gradient(90deg,#c2410c,rgba(194,65,12,0))"></div>

Truncating is not a shorter list of singular values — it is the same list with the tail set to zero. The discarded ones are drawn dashed, and the reconstruction beside them says what dropping them cost.

<img src="https://project-delphi.github.io/tensors-workshop/images/cube-09-rank.gif" alt="An animation showing a 4 by 3 matrix and its three singular values, then the rank 1, rank 2 and rank 3 reconstructions, with each discarded singular value drawn as a dashed outline." style="max-width:100%;display:block;margin:2.2em auto;border-radius:10px">

The [SVD portal](https://project-delphi.github.io/tensors-workshop/interactive/linalg-stage.html?lang=en#portal) ends on this: keep only `σ₁u₁v₁ᵀ` and you have the best rank-1 copy of `A` there is.

<div style="border-left:4px solid rgba(130,130,150,.5);background:rgba(130,130,150,.09);border-radius:0 8px 8px 0;padding:15px 19px;margin:1.8em 0 2.2em;font:400 14.5px/1.8 ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"><div style="font:700 10.5px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;opacity:.62;margin-bottom:10px">🇪🇸 ESPAÑOL</div>Truncar no es una lista más corta de valores singulares: es la misma lista con la cola puesta a cero. Los descartados se dibujan con línea discontinua, y la reconstrucción a su lado dice lo que costó dejarlos fuera. El <a href="https://project-delphi.github.io/tensors-workshop/interactive/linalg-stage.html?lang=es#portal">portal de la SVD</a> termina en esto: quédate solo con <code>σ₁u₁v₁ᵀ</code> y tienes la mejor copia de rango 1 de <code>A</code> que existe.</div>

### The factorization you can point at / La factorización que puedes señalar

<div style="height:3px;border-radius:2px;margin:1.6em 0 1.9em;background:linear-gradient(90deg,#c2410c,rgba(194,65,12,0))"></div>

SVD gives the best rank-k approximation there is, and `cube-09-rank` above says so. This is why you would ever use anything else. The data is counts, so no entry can be below zero — and a factor with negative entries describes it as one thing partly cancelled by another. True arithmetic, and not a sentence anybody can say about counts.

$$
A \approx W H,
\qquad\qquad
W \ge 0, \quad H \ge 0
$$

Read it as: the same shape of statement as `A = U S Vt`, with one constraint
added and optimality given up for it. The constraint is the whole product —
every number in both factors is a quantity of something, so a column of `W` is
a part you can name rather than a direction you can only plot.

<img src="https://project-delphi.github.io/tensors-workshop/images/cube-09-nmf.gif" alt="An animation of a 4 by 3 matrix of non-negative counts. Its SVD is shown with the negative entries of U highlighted. The same matrix is then shown as the product of a non-negative 4 by 2 matrix W and a non-negative 2 by 3 matrix H. The last frame places the first two columns of U beside W, both rank 2 on the same data, with only one of them free of negative numbers." style="max-width:100%;display:block;margin:2.2em auto;border-radius:10px">

<div style="border-left:4px solid rgba(130,130,150,.5);background:rgba(130,130,150,.09);border-radius:0 8px 8px 0;padding:15px 19px;margin:1.8em 0 2.2em;font:400 14.5px/1.8 ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"><div style="font:700 10.5px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;opacity:.62;margin-bottom:11px">🇪🇸 ESPAÑOL</div>La SVD da la mejor aproximación de rango k que existe, y <code>cube-09-rank</code> arriba lo dice. Esto explica por qué usarías otra cosa. Los datos son conteos, así que ninguna entrada puede estar por debajo de cero, y un factor con entradas negativas los describe como una cosa parcialmente cancelada por otra. Aritmética correcta, y no una frase que nadie pueda decir sobre conteos.</div>

In [ ]:
#@title ⏸️ Step through the animations / Recorre las animaciones { display-mode: 'form' }

# Plumbing, not a lesson. The animations above loop forever and a GIF cannot
# be paused — so this fetches the same frames and hands them over one at a
# time, at whatever pace you read at.
# Plomería, no una lección: trae los mismos fotogramas y los entrega de uno en
# uno, al ritmo al que leas.

import io
import urllib.request

import ipywidgets as widgets
from IPython.display import display
from PIL import Image

gif_urls = [
    "https://project-delphi.github.io/tensors-workshop/images/cube-09-svd.gif",
    "https://project-delphi.github.io/tensors-workshop/images/cube-09-rank.gif",
    "https://project-delphi.github.io/tensors-workshop/images/cube-09-nmf.gif",
]


def gif_frames(url):
    """Every frame of an animated GIF, as PNG bytes."""
    with urllib.request.urlopen(url, timeout=30) as response:
        gif = Image.open(io.BytesIO(response.read()))
    out = []
    try:
        while True:
            buffer = io.BytesIO()
            gif.convert("RGB").save(buffer, format="PNG")
            out.append(buffer.getvalue())
            gif.seek(gif.tell() + 1)
    except EOFError:
        pass
    return out


try:
    gif_cache = {url: gif_frames(url) for url in gif_urls}
except Exception as error:  # offline, or the site is down
    print("EN: could not reach the site, so there are no frames to step "
          "through.", error)
    print("ES: no se pudo acceder al sitio, así que no hay fotogramas que "
          "recorrer.", error)
else:
    gif_pick = widgets.Dropdown(
        options=[(url.rsplit("/", 1)[1], url) for url in gif_urls],
        description="Animation / Animación:",
        style={"description_width": "180px"},
    )
    gif_step = widgets.IntSlider(
        min=1, max=len(gif_cache[gif_urls[0]]), value=1,
        description="Frame / Fotograma:",
        style={"description_width": "180px"},
        continuous_update=False,
    )
    gif_prev = widgets.Button(description="◀ Prev")
    gif_next = widgets.Button(description="Next ▶")
    # An Image widget, deliberately, and never widgets.Output: a payload
    # leaving an Output widget makes nbclient wait out the whole cell timeout
    # (see scripts/test_notebooks.py). This one is a plain bytes trait.
    gif_view = widgets.Image(format="png",
                             layout=widgets.Layout(max_width="100%"))

    def gif_show(*_):
        frames = gif_cache[gif_pick.value]
        gif_step.max = len(frames)
        gif_view.value = frames[min(gif_step.value, len(frames)) - 1]

    def gif_bump(delta):
        def click(_):
            frames = gif_cache[gif_pick.value]
            gif_step.value = (gif_step.value - 1 + delta) % len(frames) + 1
        return click

    gif_prev.on_click(gif_bump(-1))
    gif_next.on_click(gif_bump(+1))
    gif_pick.observe(gif_show, names="value")
    gif_step.observe(gif_show, names="value")
    gif_show()

    display(widgets.VBox([
        gif_pick,
        widgets.HBox([gif_prev, gif_step, gif_next]),
        gif_view,
    ]))


### Interactive method chooser / Selector interactivo de método

Tick the properties your data actually has and read back which factorization fits, why, and what it costs.

The point is not to memorize the table. It is that **four or five yes/no facts about your matrix determine the answer**, and you almost always know those facts before you write any code.

> 🇪🇸 Marca las propiedades que realmente tienen tus datos y lee qué factorización encaja, por qué y cuánto cuesta.
>
> Lo importante no es memorizar la tabla, sino que **cuatro o cinco hechos de sí/no sobre tu matriz determinan la respuesta**, y casi siempre los conoces antes de escribir código.

In [ ]:
#@title Method chooser / Selector de método — run me / ejecútame { display-mode: 'form' }

square = widgets.Checkbox(value=False, description="Square / Cuadrada")
symmetric = widgets.Checkbox(value=False, description="Symmetric / Simétrica")
posdef = widgets.Checkbox(
    value=False, description="Positive definite / Definida positiva")
sparse = widgets.Checkbox(value=False, description="Sparse / Dispersa")
nonneg = widgets.Checkbox(
    value=False, description="Nonnegative / No negativa")
readable = widgets.Checkbox(
    value=False, description="Factors must be readable / Factores legibles")
low_rank = widgets.Checkbox(
    value=False, description="Only top k needed / Solo las primeras k")
many_rhs = widgets.Checkbox(
    value=False, description="Many right-hand sides / Muchos lados derechos")

def choose(square, symmetric, posdef, sparse, nonneg,
           readable, low_rank, many_rhs):
    # Ordered most-specific first: the first rule that fires wins, which is
    # how a practitioner actually decides.
    if nonneg and readable:
        pick = "NMF"
        why_en = ("nonnegative data plus a demand for readable factors is the "
                  "one case where giving up optimal error is the right trade.")
        why_es = ("datos no negativos más la exigencia de factores legibles: "
                  "el único caso donde renunciar al error óptimo compensa.")
        cost = "O(mnk) per iteration / por iteración"
    elif low_rank and sparse:
        pick = "Lanczos / truncated SVD (scipy.sparse.linalg.svds)"
        why_en = ("a sparse matrix should never be densified; Lanczos touches "
                  "only the nonzeros.")
        why_es = ("una matriz dispersa nunca debe densificarse; Lanczos toca "
                  "solo los no ceros.")
        cost = "O(k · nnz(A)) per restart / por reinicio"
    elif low_rank:
        pick = "Randomized SVD (sklearn.utils.extmath.randomized_svd)"
        why_en = ("you asked for k components, so do work proportional to k "
                  "rather than to min(m, n).")
        why_es = ("pediste k componentes: haz trabajo proporcional a k y no a "
                  "min(m, n).")
        cost = "O(mnk)"
    elif symmetric and posdef and many_rhs:
        pick = "Cholesky (scipy.linalg.cho_factor / cho_solve)"
        why_en = ("half the flops of LU, and the factorization is reused by "
                  "every right-hand side.")
        why_es = ("la mitad de flops que LU, y la factorización se reutiliza "
                  "en cada lado derecho.")
        cost = "O(n³/3) once, then O(n²) per solve / luego por solución"
    elif symmetric and posdef:
        pick = "Cholesky (np.linalg.cholesky)"
        why_en = "symmetry and positive definiteness halve the work; use them."
        why_es = ("la simetría y la definición positiva reducen el trabajo a "
                  "la mitad; aprovéchalas.")
        cost = "O(n³/3)"
    elif symmetric:
        pick = "Eigendecomposition (np.linalg.eigh)"
        why_en = ("a symmetric matrix has a real orthogonal eigenbasis — eigh "
                  "exploits it, eig does not.")
        why_es = ("una matriz simétrica tiene base propia ortogonal real: eigh "
                  "lo aprovecha, eig no.")
        cost = "O(9n³) with eigenvectors / con autovectores"
    elif square and many_rhs:
        pick = "LU (scipy.linalg.lu_factor / lu_solve)"
        why_en = ("factor once, then every extra solve is a pair of triangular "
                  "substitutions.")
        why_es = ("factoriza una vez y cada solución extra son dos "
                  "sustituciones triangulares.")
        cost = "O(2n³/3) once, then O(n²) per solve / luego por solución"
    elif square:
        pick = "LU (np.linalg.solve)"
        why_en = "np.linalg.solve is LU with pivoting, and never forms an inverse."
        why_es = ("np.linalg.solve es LU con pivoteo y nunca forma una "
                  "inversa.")
        cost = "O(2n³/3)"
    else:
        pick = "QR (np.linalg.qr, or np.linalg.lstsq)"
        why_en = ("a tall matrix means least squares, and QR solves it at "
                  "condition number κ(X) rather than κ(X)².")
        why_es = ("una matriz alta significa mínimos cuadrados, y QR lo "
                  "resuelve con número de condición κ(X) y no κ(X)².")
        cost = "O(2mn² − 2n³/3)"

    print("Use / Usa:", pick)
    print("Cost / Coste:", cost)
    print()
    print("EN:", why_en)
    print("ES:", why_es)

    if posdef and not symmetric:
        print()
        print("EN: note — positive definite is only defined for symmetric "
              "matrices here; tick symmetric too.")
        print("ES: nota — aquí la definición positiva solo tiene sentido para "
              "matrices simétricas; marca también simétrica.")

chooser_output = widgets.interactive_output(
    choose,
    {
        "square": square,
        "symmetric": symmetric,
        "posdef": posdef,
        "sparse": sparse,
        "nonneg": nonneg,
        "readable": readable,
        "low_rank": low_rank,
        "many_rhs": many_rhs,
    },
)

display(widgets.VBox([
    widgets.HBox([square, symmetric, posdef, sparse]),
    widgets.HBox([nonneg, readable, low_rank, many_rhs]),
    chooser_output,
]))

### When the assumption fails / Cuando el supuesto no se cumple

A method can still return an answer after an assumption fails. That answer may be the wrong tool.

Predict the result of each call.

- Cholesky rejects a matrix that is not positive definite.
- NMF rejects negative data.
- `eigh` assumes symmetry. It uses the lower triangle by default and can silently solve a different problem.

<div style="border-left:4px solid rgba(130,130,150,.5);background:rgba(130,130,150,.09);border-radius:0 8px 8px 0;padding:15px 19px;margin:1.8em 0 2.2em;font:400 14.5px/1.8 ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"><div style="font:700 10.5px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;opacity:.62;margin-bottom:10px">🇪🇸 ESPAÑOL</div>Predice cada resultado. Cholesky exige definición positiva; NMF exige datos no negativos. `eigh` supone simetría y usa el triángulo inferior por defecto: puede resolver otro problema sin avisar.</div>

In [ ]:
# 1. Cholesky needs positive definiteness, and says so.
pixel_covariance = np.cov(D[:, :20].T)
smallest = np.linalg.eigvalsh(pixel_covariance).min()

print("Covariance of 20 real digit pixels / Covarianza de 20 píxeles reales:",
      pixel_covariance.shape)
print("Constant pixels among them / Píxeles constantes entre ellos:",
      int(np.sum(D[:, :20].var(axis=0) == 0)))
print("Smallest eigenvalue / Menor autovalor:", f"{smallest:.3e}")
print("Rank / Rango:", np.linalg.matrix_rank(pixel_covariance), "of / de 20")
try:
    np.linalg.cholesky(pixel_covariance)
    print("Cholesky succeeded / Cholesky funcionó")
except np.linalg.LinAlgError as exc:
    print("Cholesky raised / Cholesky lanzó:", exc)

# 2. NMF needs nonnegativity, and says so.
try:
    NMF(n_components=2, max_iter=10).fit(D - 0.5)
except ValueError as exc:
    print()
    print("NMF raised / NMF lanzó:", str(exc).split(".")[0])

# 3. eigh needs symmetry, and does NOT say so.
block = D[10:14, 20:24]
eig_values = np.linalg.eig(block)[0]

print()
print("A real 4x4 pixel block / Un bloque real de 4x4 píxeles, symmetric?",
      np.allclose(block, block.T))
# eig returns a complex array for any non-symmetric input. Here the largest
# imaginary part is zero, so dropping it is safe — but check before you do,
# because a non-symmetric matrix is exactly the kind that can have genuinely
# complex eigenvalues (a rotation matrix has a pair of them).
print("Largest imaginary part / Mayor parte imaginaria:",
      f"{np.abs(eig_values.imag).max():.1e}")
print("eig  (correct / correcto) :",
      np.round(np.sort(eig_values.real), 4))
print("eigh (silently wrong / silenciosamente incorrecto):",
      np.round(np.sort(np.linalg.eigh(block)[0]), 4))
print()
print("What eigh actually decomposed / Lo que eigh descompuso de verdad:")
print(np.round(np.tril(block) + np.tril(block, -1).T, 4))
print()
print("EN: eigh mirrored the lower triangle and factorized THAT. Nothing in")
print("    the call signalled it. Check symmetry yourself:")
print("        assert np.allclose(A, A.T)")
print("ES: eigh reflejó el triángulo inferior y factorizó ESO. Nada en la")
print("    llamada lo indicó. Comprueba la simetría tú mismo:")
print("        assert np.allclose(A, A.T)")

## 9.2 The cost model / El modelo de coste

<div style="height:3px;border-radius:2px;margin:1.6em 0 1.9em;background:linear-gradient(90deg,#c2410c,rgba(194,65,12,0))"></div>

Cost is a growth rate, not a stopwatch. Use it to rule out methods before timing.

For dense `m × n` matrices, `m ≥ n`, Setup uses these estimates:

| Method / Método | Estimated flops / Operaciones estimadas |
|---|---|
| Cholesky, `n × n` | `n³/3` |
| LU, `n × n` | `2n³/3` |
| Householder QR | `2mn² − 2n³/3` |
| Symmetric eigenvectors / Autovectores simétricos | `≈ 9n³` |
| Thin SVD / SVD reducida | `≈ 2mn² + 11n³` |
| Randomized SVD / SVD aleatorizada | `≈ 4mnk` (simplified model / modelo simplificado) |

Constants depend on the algorithm and requested outputs.

Factor once, solve `q` right-hand sides: `O(n³ + qn²)`.

For full column rank, `κ₂(XᵀX) = κ₂(X)²`. QR avoids forming `XᵀX`.

<div style="border-left:4px solid rgba(130,130,150,.5);background:rgba(130,130,150,.09);border-radius:0 8px 8px 0;padding:15px 19px;margin:1.8em 0 2.2em;font:400 14.5px/1.8 ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"><div style="font:700 10.5px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;opacity:.62;margin-bottom:10px">🇪🇸 ESPAÑOL</div>Las constantes dependen del algoritmo. Factorizar una vez permite resolver `q` lados derechos por `O(n³ + qn²)`. Las ecuaciones normales elevan al cuadrado el número de condición.</div>

### Interactive conditioning explorer / Explorador interactivo de condicionamiento

Slide the polynomial degree and watch four numbers move together on the real airline data:

- `κ(X)` — the condition number of the design matrix;
- `κ(XᵀX)` — which tracks `κ(X)²`, not `κ(X)`;
- the coefficient error of the **normal equations**;
- the coefficient error of **QR**.

Degree 4 is harmless. By degree 10 the normal equations have lost eleven digits that QR still has. By degree 12 they return a visibly worse *fit*, not just worse coefficients. The residual itself degrades, which is where even a residual-only check would finally notice.

The stage draws both halves of this. Its [collinear step](https://project-delphi.github.io/tensors-workshop/interactive/linalg-stage.html?lang=en#collinear) closes the angle between two columns and shows `β` walking far out and far back while `ŷ` stays put; its [float32 step](https://project-delphi.github.io/tensors-workshop/interactive/linalg-stage.html?lang=en#precision) closes the same angle until the two stored columns become one vector and the normal equations divide by exactly zero.

> 🇪🇸 Desliza el grado del polinomio y observa cómo se mueven juntos cuatro números sobre los datos reales de aerolíneas. Son `κ(X)`, `κ(XᵀX)` — que sigue a `κ(X)²` — y el error de coeficientes de las ecuaciones normales y de QR.
>
> El grado 4 es inofensivo. En el grado 10 las ecuaciones normales han perdido once dígitos que QR conserva. En el grado 12 empeora el propio residuo, que es cuando incluso una comprobación basada solo en residuos se daría cuenta.
>
> El escenario dibuja las dos mitades de esto. Su [paso colineal](https://project-delphi.github.io/tensors-workshop/interactive/linalg-stage.html?lang=es#collinear) cierra el ángulo entre dos columnas y muestra a `β` yendo muy lejos y volviendo mientras `ŷ` no se mueve; su [paso float32](https://project-delphi.github.io/tensors-workshop/interactive/linalg-stage.html?lang=es#precision) cierra el mismo ángulo hasta que las dos columnas guardadas se vuelven un solo vector y las ecuaciones normales dividen exactamente por cero.


In [ ]:
#@title Conditioning explorer / Explorador de condicionamiento — run me / ejecútame { display-mode: 'form' }

# The degree-10 design matrix and target (Exercise 1, TODO 1 step 1), rebound
# here so this explorer runs whether or not the folded solution was executed.
X = np.vander(month_scaled, 11, increasing=True)
y = passengers

degree_slider = widgets.IntSlider(
    value=10,
    min=2,
    max=14,
    step=1,
    description="Degree / Grado:",
    continuous_update=False,
    style={"description_width": "140px"},
)

def compare_conditioning(degree):
    Xd = np.vander(month_scaled, degree + 1, increasing=True)

    cond_X = np.linalg.cond(Xd)
    cond_normal = np.linalg.cond(Xd.T @ Xd)

    beta_normal = np.linalg.solve(Xd.T @ Xd, Xd.T @ y)
    Qd, Rd = np.linalg.qr(Xd)
    beta_qr = np.linalg.solve(Rd, Qd.T @ y)
    beta_ref = np.linalg.lstsq(Xd, y, rcond=None)[0]

    err = lambda b: (np.linalg.norm(b - beta_ref)
                     / np.linalg.norm(beta_ref))
    residual = lambda b: np.linalg.norm(Xd @ b - y)

    print("Design matrix / Matriz de diseño:", Xd.shape)
    print("cond(X)   =", f"{cond_X:.2e}")
    print("cond(XtX) =", f"{cond_normal:.2e}",
          "   cond(X)^2 =", f"{cond_X ** 2:.2e}")
    print()
    print("Coefficient error / Error de coeficientes:")
    print("   normal equations / ecuaciones normales:", f"{err(beta_normal):.2e}")
    print("   QR                                    :", f"{err(beta_qr):.2e}")
    print()
    print("Residual / Residuo:")
    print("   normal equations / ecuaciones normales:", f"{residual(beta_normal):.4f}")
    print("   QR                                    :", f"{residual(beta_qr):.4f}")

    fig, ax = plt.subplots(figsize=(7.5, 3.2))
    ax.plot(month, y, ".", color="0.55", markersize=4,
            label="Real passengers / Pasajeros reales")
    ax.plot(month, Xd @ beta_qr, "-", linewidth=2, label="QR")
    ax.plot(month, Xd @ beta_normal, "--", linewidth=1.6,
            label="Normal equations / Ecuaciones normales")
    ax.set_xlabel("Month index / Índice de mes")
    ax.set_ylabel("Passengers / Pasajeros")
    ax.set_title(f"Degree {degree} polynomial fit / Ajuste polinómico de grado {degree}")
    ax.legend(fontsize=8)
    plt.show()

conditioning_output = widgets.interactive_output(
    compare_conditioning,
    {"degree": degree_slider},
)

display(widgets.VBox([degree_slider, conditioning_output]))

## Exercise 2 — factor once, solve many / Ejercicio 2 — factoriza una vez, resuelve muchas

<div style="height:3px;border-radius:2px;margin:1.6em 0 1.9em;background:linear-gradient(90deg,#c2410c,rgba(194,65,12,0))"></div>

Factor once when the matrix stays fixed and many right-hand sides arrive.

Build the `1797 × 1797` digit RBF kernel. Add a positive ridge term to its diagonal.

Time 200 right-hand sides: reuse Cholesky, compute `inv(G) @ B`, and call `solve` per column.

**Predict:** Which method repeats the expensive step? Compare timings and relative residuals.

<div style="border-left:4px solid rgba(130,130,150,.5);background:rgba(130,130,150,.09);border-radius:0 8px 8px 0;padding:15px 19px;margin:1.8em 0 2.2em;font:400 14.5px/1.8 ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"><div style="font:700 10.5px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;opacity:.62;margin-bottom:10px">🇪🇸 ESPAÑOL</div>Construye y regulariza el kernel de los dígitos. Compara Cholesky reutilizada, inversa explícita y `solve` por columna para 200 lados derechos. Predice qué método repite el paso caro; mide tiempos y residuos.</div>

In [ ]:
# TODO 2 / TAREA 2
#
# EN:
# 1. Build the RBF kernel matrix of the digits and add a ridge term:
#       sq = np.sum(D ** 2, axis=1)
#       D2 = sq[:, None] + sq[None, :] - 2 * D @ D.T
#       G  = np.exp(-0.05 * np.maximum(D2, 0)) + 1e-6 * np.eye(len(D))
#    Print G.shape and np.linalg.cond(G).
# 2. Build 200 right-hand sides: B = rng.standard_normal((len(G), 200)).
# 3. Time, with best_time(...):
#       a) cho_solve(cho_factor(G), B)                 <- factor once
#       b) np.linalg.inv(G) @ B                        <- explicit inverse
#       c) one np.linalg.solve(G, B[:, j]) per column  <- refactor each time
#    Use repeats=1 for (c); it is slow on purpose.
# 4. Print each time and the ratio against (a).
# 5. Print the relative residual ||G @ Xhat - B|| / ||B|| for (a) and (b).
#    Which is more accurate?
# 6. Bonus: solve against one-hot digit labels instead of random noise and
#    report the training accuracy of the resulting kernel-ridge classifier.
#
# ES:
# 1. Construye la matriz kernel RBF de los dígitos más un término ridge.
# 2. Crea 200 lados derechos aleatorios.
# 3. Cronometra las tres estrategias (usa repeats=1 en la tercera).
# 4. Imprime cada tiempo y la razón frente a la primera.
# 5. Imprime el residuo relativo de las dos primeras. ¿Cuál es más precisa?
# 6. Extra: resuelve contra etiquetas one-hot y reporta la exactitud.

In [ ]:
#@title Solution / Solución — try it yourself first / inténtalo primero { display-mode: 'form' }

sq = np.sum(D ** 2, axis=1)
D2 = sq[:, None] + sq[None, :] - 2 * D @ D.T
G = np.exp(-0.05 * np.maximum(D2, 0)) + 1e-6 * np.eye(len(D))

n_rhs = 200
B = rng.standard_normal((len(G), n_rhs))

print("Kernel system / Sistema kernel:", G.shape)
print("cond(G) =", f"{np.linalg.cond(G):.3e}")
print("Right-hand sides / Lados derechos:", n_rhs)
print()

t_factor = best_time(lambda: cho_solve(cho_factor(G), B))
t_inverse = best_time(lambda: np.linalg.inv(G) @ B)
t_percol = best_time(
    lambda: np.column_stack(
        [np.linalg.solve(G, B[:, j]) for j in range(n_rhs)]),
    repeats=1,
)

print("Factor once + 200 solves / Factorizar una vez + 200 soluciones:",
      f"{t_factor * 1e3:9.1f} ms   (1.0x)")
print("Explicit inverse / Inversa explícita:                          ",
      f"{t_inverse * 1e3:9.1f} ms   ({t_inverse / t_factor:.1f}x)")
print("One solve per column / Una solución por columna:               ",
      f"{t_percol * 1e3:9.1f} ms   ({t_percol / t_factor:.1f}x)")
print()

X_factor = cho_solve(cho_factor(G), B)
X_inverse = np.linalg.inv(G) @ B
rel = lambda Xh: np.linalg.norm(G @ Xh - B) / np.linalg.norm(B)

print("Relative residual / Residuo relativo:")
print("   factor once / factorizar una vez:", f"{rel(X_factor):.3e}")
print("   explicit inverse / inversa explícita:", f"{rel(X_inverse):.3e}")
print()

# Score it held out: with this little ridge the solve interpolates, so a
# training accuracy of 1.0 would say nothing about the model.
holdout = np.random.default_rng(7).permutation(len(D))
fit_idx, score_idx = holdout[:1200], holdout[1200:]
alpha = cho_solve(cho_factor(G[np.ix_(fit_idx, fit_idx)]),
                  np.eye(10)[digit_labels[fit_idx]])
predicted = (G[np.ix_(score_idx, fit_idx)] @ alpha).argmax(axis=1)
accuracy = np.mean(predicted == digit_labels[score_idx])
print("Kernel-ridge held-out accuracy / Exactitud reservada:",
      f"{accuracy:.4f}", f"on / sobre {len(score_idx)} unseen digits")
print()
print("EN: one factorization amortized over 200 solves is O(n^3 + m n^2).")
print("    Refactoring per column is O(m n^3) and the measured ratio shows it.")
print("    The explicit inverse is both slower and less accurate — it is never")
print("    the right call.")
print("ES: una factorización amortizada en 200 soluciones es O(n^3 + m n^2);")
print("    refactorizar por columna es O(m n^3), y la razón medida lo muestra.")
print("    La inversa explícita es más lenta Y menos precisa: nunca es la")
print("    llamada correcta.")

### Interactive repeated-solve explorer / Explorador interactivo de resolución repetida

Watch setup cost separately from each solve. Reuse only pays when enough solves follow.

Move the number of right-hand sides.

The per-column curve is measured through 20 columns. Beyond that, it is a linear extrapolation.

<div style="border-left:4px solid rgba(130,130,150,.5);background:rgba(130,130,150,.09);border-radius:0 8px 8px 0;padding:15px 19px;margin:1.8em 0 2.2em;font:400 14.5px/1.8 ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"><div style="font:700 10.5px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;opacity:.62;margin-bottom:10px">🇪🇸 ESPAÑOL</div>Cambia el número de lados derechos. La curva por columna se mide hasta 20; después se extrapola linealmente.</div>

In [ ]:
#@title Repeated solves / Resoluciones repetidas — run me / ejecútame { display-mode: 'form' }

# The real kernel system (Exercise 2, TODO 2 step 1), rebound here so this
# explorer runs whether or not the folded solution was executed.
sq = np.sum(D ** 2, axis=1)
D2 = sq[:, None] + sq[None, :] - 2 * D @ D.T
G = np.exp(-0.05 * np.maximum(D2, 0)) + 1e-6 * np.eye(len(D))

rhs_grid = np.array([1, 2, 5, 10, 20, 50, 100, 200])
MEASURED_PER_COLUMN_UP_TO = 20

t_factor_once, t_inverse_grid, t_per_column = [], [], []

for m_rhs in rhs_grid:
    B = rng.standard_normal((len(G), int(m_rhs)))
    t_factor_once.append(best_time(lambda: cho_solve(cho_factor(G), B)))
    t_inverse_grid.append(best_time(lambda: np.linalg.inv(G) @ B))
    if m_rhs <= MEASURED_PER_COLUMN_UP_TO:
        t_per_column.append(best_time(
            lambda: np.column_stack(
                [np.linalg.solve(G, B[:, j]) for j in range(int(m_rhs))]),
            repeats=1,
        ))

t_factor_once = np.array(t_factor_once)
t_inverse_grid = np.array(t_inverse_grid)

# Refactoring per column is exactly linear in m, so one slope extends it.
per_column_slope = (np.array(t_per_column)
                    / rhs_grid[:len(t_per_column)]).mean()
t_per_column = per_column_slope * rhs_grid

print("Measured on / Medido sobre:", G.shape)
print("Per-column cost per solve / Coste por solución:",
      f"{per_column_slope * 1e3:.1f} ms")
print()

rhs_slider = widgets.SelectionSlider(
    options=[int(m_rhs) for m_rhs in rhs_grid],
    value=200,
    description="Right-hand sides / Lados derechos:",
    continuous_update=False,
    style={"description_width": "220px"},
)

def show_repeated_solve(m_rhs):
    i = int(np.where(rhs_grid == m_rhs)[0][0])
    base = t_factor_once[i]

    print(f"m = {m_rhs} right-hand sides / lados derechos")
    print("Factor once / Factorizar una vez:  ",
          f"{base * 1e3:9.1f} ms   (1.0x)")
    print("Explicit inverse / Inversa explícita:",
          f"{t_inverse_grid[i] * 1e3:9.1f} ms   "
          f"({t_inverse_grid[i] / base:.1f}x)")
    print("Refactor per column / Refactorizar por columna:",
          f"{t_per_column[i] * 1e3:9.1f} ms   "
          f"({t_per_column[i] / base:.1f}x)"
          + ("" if m_rhs <= MEASURED_PER_COLUMN_UP_TO
             else "   [extrapolated / extrapolado]"))

    fig, ax = plt.subplots(figsize=(7.0, 3.4))
    ax.loglog(rhs_grid, t_factor_once * 1e3, "o-",
              label="Factor once / Factorizar una vez")
    ax.loglog(rhs_grid, t_inverse_grid * 1e3, "s-",
              label="Explicit inverse / Inversa explícita")
    ax.loglog(rhs_grid, t_per_column * 1e3, "^--",
              label="Refactor per column / Por columna")
    ax.axvline(m_rhs, color="0.6", linewidth=1)
    ax.set_xlabel("Right-hand sides m / Lados derechos m")
    ax.set_ylabel("Time (ms) / Tiempo (ms)")
    ax.set_title("Cost of m solves on one real 1797x1797 SPD system")
    ax.legend(fontsize=8)
    ax.grid(True, which="both", alpha=0.25)
    plt.show()

repeated_output = widgets.interactive_output(
    show_repeated_solve,
    {"m_rhs": rhs_slider},
)

display(widgets.VBox([rhs_slider, repeated_output]))

## 9.3 The cost table, measured / La tabla de costes, medida

<div style="height:3px;border-radius:2px;margin:1.6em 0 1.9em;background:linear-gradient(90deg,#c2410c,rgba(194,65,12,0))"></div>

Measure a slope from several sizes. One timing is a story; a trend is evidence.

If `t(n) = Cnᵖ`, then `log t = log C + p log n`.

$$
t(n) = C n^{p}
\qquad\Longrightarrow\qquad
\log t = \log C + p \log n
$$


The log-log slope estimates `p`. The next cell creates `sweep_sizes` and `sweep_times` for Exercise 3.

These dense square factorizations have cubic arithmetic. Timed slopes also reflect overhead, caching, and parallelism.

<div style="border-left:4px solid rgba(130,130,150,.5);background:rgba(130,130,150,.09);border-radius:0 8px 8px 0;padding:15px 19px;margin:1.8em 0 2.2em;font:400 14.5px/1.8 ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"><div style="font:700 10.5px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;opacity:.62;margin-bottom:10px">🇪🇸 ESPAÑOL</div>La pendiente log-log estima `p`. La celda crea los datos del ejercicio 3. La aritmética es cúbica; los tiempos también dependen de caché, paralelismo y costes fijos.</div>

In [ ]:
# The size sweep. On a Colab CPU this takes roughly half a minute; drop the
# largest size if you are impatient.
sweep_sizes = np.array([128, 192, 256, 384, 512, 768])

# Its own generator, so re-running any cell above cannot change the matrices
# this sweep is timed on — the quoted ratios depend on them.
sweep_rng = np.random.default_rng(1)

def make_spd(n):
    """A symmetric positive-definite n x n matrix, so every method applies."""
    A = sweep_rng.standard_normal((n, n))
    return A @ A.T + n * np.eye(n)

sweep_matrices = {int(n): make_spd(int(n)) for n in sweep_sizes}

SWEEP_METHODS = {
    "Cholesky": np.linalg.cholesky,
    "LU": lu_factor,
    "QR": np.linalg.qr,
    "Eigendecomposition": np.linalg.eigh,
    "Thin SVD": lambda A: np.linalg.svd(A, full_matrices=False),
}

sweep_times = {}
for name, call in SWEEP_METHODS.items():
    sweep_times[name] = np.array([
        best_time(lambda: call(sweep_matrices[int(n)])) for n in sweep_sizes
    ])

print("Sizes / Tamaños:", [int(n) for n in sweep_sizes])
print()
header = "method / método (ms)".ljust(22) + "".join(
    f"{int(n):>10d}" for n in sweep_sizes)
print(header)
print("-" * len(header))
for name, times in sweep_times.items():
    print(name.ljust(22)
          + "".join(f"{t * 1e3:10.2f}" for t in times))
print()
largest = int(sweep_sizes[-1])
ratio = (sweep_times["Thin SVD"][-1] / sweep_times["Cholesky"][-1])
print(f"At n = {largest}: SVD / Cholesky = {ratio:.1f}x  "
      f"(flop table predicts / la tabla predice 39x)")

## Exercise 3 — fit the exponent / Ejercicio 3 — ajusta el exponente

You now hold `sweep_sizes` and `sweep_times`. Turn them into exponents.

### Predict before you run / Predice antes de ejecutar

Which method's fitted slope will be *closest* to 3, and why? Think about which one does the most arithmetic per byte moved — that is the one whose timing is least polluted by memory traffic.

> 🇪🇸 Ya tienes `sweep_sizes` y `sweep_times`. Conviértelos en exponentes.
>
> **Predice antes de ejecutar:** ¿qué método tendrá la pendiente más cercana a 3 y por qué? Piensa cuál hace más aritmética por byte movido: ese es el que menos contamina el tráfico de memoria.

In [ ]:
# TODO 3 / TAREA 3
#
# EN:
# 1. For each method in sweep_times, fit a line through the log-log points:
#       slope, intercept = np.polyfit(np.log(sweep_sizes), np.log(times), 1)
#    The slope is the measured exponent.
# 2. Print the fitted exponent beside the predicted one. Every dense
#    factorization here is cubic, so the predicted exponent is 3 for all five.
# 3. Draw a log-log plot: the measured points, the fitted line, and a
#    reference line proportional to n**3 through the first point.
# 4. Compute the measured cost ratio SVD / Cholesky at the largest size and
#    compare it against the flop-table prediction of 39x.
# 5. Answer in one sentence: which is the more trustworthy prediction from
#    the flop table on this machine, the exponent or the ratio?
#
# ES:
# 1. Ajusta una recta a los puntos log-log de cada método; la pendiente es
#    el exponente medido.
# 2. Imprime el exponente ajustado junto al predicho (3 para los cinco).
# 3. Dibuja un gráfico log-log con los puntos, la recta ajustada y una
#    referencia proporcional a n**3.
# 4. Calcula la razón medida SVD / Cholesky en el mayor tamaño y compárala
#    con las 39x que predice la tabla de flops.
# 5. Responde en una frase: en esta máquina, ¿qué predice mejor la tabla de
#    flops, el exponente o la razón?

In [ ]:
#@title Solution / Solución — try it yourself first / inténtalo primero { display-mode: 'form' }

log_n = np.log(sweep_sizes)

print("method / método".ljust(20), "fitted / ajustado", " predicted / predicho")
print("-" * 60)
fitted = {}
for name, times in sweep_times.items():
    slope, intercept = np.polyfit(log_n, np.log(times), 1)
    fitted[name] = (slope, intercept)
    print(name.ljust(20), f"{slope:16.2f}", f"{3.0:20.1f}")

fig, ax = plt.subplots(figsize=(7.5, 4.2))
for name, times in sweep_times.items():
    slope, intercept = fitted[name]
    line = ax.loglog(sweep_sizes, times * 1e3, "o", label=f"{name} (p={slope:.2f})")
    ax.loglog(sweep_sizes, np.exp(intercept) * sweep_sizes ** slope * 1e3,
              "-", linewidth=1, color=line[0].get_color())

reference = sweep_times["Thin SVD"][0] * (sweep_sizes / sweep_sizes[0]) ** 3
ax.loglog(sweep_sizes, reference * 1e3, "k:", linewidth=1.5,
          label="slope 3 reference / referencia pendiente 3")
ax.set_xlabel("Matrix size n / Tamaño n")
ax.set_ylabel("Time (ms) / Tiempo (ms)")
ax.set_title("Measured cost against size / Coste medido frente al tamaño")
ax.legend(fontsize=8)
ax.grid(True, which="both", alpha=0.25)
plt.show()

measured_ratio = sweep_times["Thin SVD"][-1] / sweep_times["Cholesky"][-1]
print()
print(f"SVD / Cholesky at n = {int(sweep_sizes[-1])}: "
      f"measured / medido {measured_ratio:.1f}x, predicted / predicho 39x")
print()
print("EN: the fitted exponents come in under 3 because parallelism and cache")
print("    reuse both improve as n grows — the machine gets faster at the same")
print("    work. The RATIO between methods is the more trustworthy prediction:")
print("    it cancels the machine out, because both methods gain from the same")
print("    hardware effects.")
print("ES: los exponentes ajustados quedan por debajo de 3 porque el")
print("    paralelismo y la caché mejoran al crecer n: la máquina se vuelve más")
print("    eficiente con el mismo trabajo. La RAZÓN entre métodos predice")
print("    mejor, porque cancela la máquina: ambos métodos ganan lo mismo.")

## 9.4 SVD — the best rank-k there is / SVD — la mejor aproximación de rango k

<div style="height:3px;border-radius:2px;margin:1.6em 0 1.9em;background:linear-gradient(90deg,#c2410c,rgba(194,65,12,0))"></div>

SVD gives the best rank-k approximation in Frobenius norm. “Best” has that precise meaning.

Keep the largest $k$ of the singular values and throw the rest away:

$$
A = U\Sigma V^{\mathsf T}
\qquad
A_k = \sum_{i=1}^{k} \sigma_i u_i v_i^{\mathsf T}
\qquad
\bigl\lVert A - A_k \bigr\rVert_F = \sqrt{\sum_{i>k} \sigma_i^{2}}
$$

Read it as: the third equation is Eckart–Young–Mirsky, and it is what makes
this worth knowing — the error of the best rank-$k$ approximation is read
straight off the singular values you threw away, without rebuilding anything.

Check that equality on the image below. The discarded singular values give the error without rebuilding the image.

<div style="border-left:4px solid rgba(130,130,150,.5);background:rgba(130,130,150,.09);border-radius:0 8px 8px 0;padding:15px 19px;margin:1.8em 0 2.2em;font:400 14.5px/1.8 ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"><div style="font:700 10.5px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;opacity:.62;margin-bottom:10px">🇪🇸 ESPAÑOL</div>Conserva los primeros `k` términos. Comprueba la fórmula: los valores singulares descartados dan el error sin reconstruir la imagen.</div>

In [ ]:
image = IMAGES["astronaut"]

U_img, S_img, Vt_img = np.linalg.svd(image, full_matrices=False)

def truncate(k, U=U_img, S=S_img, Vt=Vt_img):
    """The rank-k truncation A_k = sum_{i<k} sigma_i u_i v_i^T."""
    return (U[:, :k] * S[:k]) @ Vt[:k]

print("Image / Imagen:", image.shape,
      " singular values / valores singulares:", S_img.shape[0])
print()
print(" k   ||A - A_k||_F    sqrt(tail)       difference / diferencia")
print("-" * 66)
for k in (5, 16, 20, 50, 100):
    built = np.linalg.norm(image - truncate(k))
    tail = np.sqrt(np.sum(S_img[k:] ** 2))
    print(f"{k:3d}   {built:13.9f}   {tail:13.9f}   {abs(built - tail):.3e}")

print()
print("EN: the two columns agree to within 1e-13. You never have to build")
print("    A_k to know how good it would be — the singular values said so.")
print("ES: las dos columnas coinciden dentro de 1e-13. Nunca hace falta")
print("    construir A_k para saberlo: los valores singulares ya lo")
print("    dijeron.")

### Interactive rank explorer on a real image / Explorador interactivo de rango sobre una imagen real

Rank controls a trade: fewer stored numbers, more reconstruction error. Pick the smallest rank that meets the requirement.

Move the rank. Compare error, PSNR, and storage.

Factors store `k(m+n+1)` values; the image stores `mn`. Multiply by bytes per value: `float32` takes four, `uint8` one.

The byte estimates exclude file overhead. Plotted quality uses unquantized factors.

<div style="border-left:4px solid rgba(130,130,150,.5);background:rgba(130,130,150,.09);border-radius:0 8px 8px 0;padding:15px 19px;margin:1.8em 0 2.2em;font:400 14.5px/1.8 ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"><div style="font:700 10.5px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;opacity:.62;margin-bottom:10px">🇪🇸 ESPAÑOL</div>Cambia el rango. Compara error, PSNR y almacenamiento. Los factores guardan `k(m+n+1)` valores. Multiplica por bytes por valor: cuatro para `float32`, uno para `uint8`. Los bytes excluyen cabeceras; la calidad usa factores sin cuantizar.</div>

In [ ]:
#@title Rank explorer / Explorador de rango — run me / ejecútame { display-mode: 'form' }

image_choice = widgets.ToggleButtons(
    options=[("Astronaut", "astronaut"), ("Camera", "camera")],
    value="astronaut",
    description="Image / Imagen:",
    style={"description_width": "130px"},
)

rank_slider = widgets.IntSlider(
    value=16,
    min=1,
    max=200,
    step=1,
    description="Rank k / Rango k:",
    continuous_update=False,
    style={"description_width": "130px"},
)

# One SVD per image, computed once so the slider stays instant.
IMAGE_SVD = {name: np.linalg.svd(A, full_matrices=False)
             for name, A in IMAGES.items()}

def explore_rank(which, k):
    A = IMAGES[which]
    U, S, Vt = IMAGE_SVD[which]
    m, n = A.shape

    Ak = (U[:, :k] * S[:k]) @ Vt[:k]

    stored = k * (m + n + 1)
    energy = np.sum(S[:k] ** 2) / np.sum(S ** 2)

    print("Rank / Rango:", k, " of / de", min(m, n))
    print("Relative error / Error relativo:", f"{relative_error(A, Ak):.4f}")
    print("PSNR:", f"{psnr(A, Ak):.2f} dB")
    print("Energy kept / Energía conservada:", f"{100 * energy:.3f}%")
    print()
    print("Numbers stored / Números almacenados:", f"{stored:,}",
          "of / de", f"{m * n:,}", f"({100 * stored / (m * n):.2f}%)")
    print("Bytes as int16 / Bytes en int16:",
          f"{100 * stored * 2 / (m * n):.1f}% of the raw uint8 image")
    print("Bytes as float32 / Bytes en float32:",
          f"{100 * stored * 4 / (m * n):.1f}% of the raw uint8 image")

    fig, axes = plt.subplots(1, 3, figsize=(10.5, 3.6))
    axes[0].imshow(A, cmap="gray", vmin=0, vmax=1)
    axes[0].set_title("Original")
    axes[1].imshow(Ak, cmap="gray", vmin=0, vmax=1)
    axes[1].set_title(f"Rank {k} / Rango {k}")
    axes[2].imshow(np.abs(A - Ak), cmap="magma")
    axes[2].set_title("|difference| / |diferencia|")
    for ax in axes:
        ax.axis("off")
    plt.tight_layout()
    plt.show()

rank_output = widgets.interactive_output(
    explore_rank,
    {"which": image_choice, "k": rank_slider},
)

display(widgets.VBox([image_choice, rank_slider, rank_output]))

## Exercise 4 — the rank that crosses a threshold / Ejercicio 4 — el rango que cruza un umbral

"Rank 20" is not a decision anybody can defend. "The smallest rank that holds 25 dB" is.

Turn the requirement round: pick a quality target, then find the cheapest rank that meets it — and report what that rank actually costs to store.

### Predict before you run / Predice antes de ejecutar

Each extra 6 dB is roughly a halving of the error. Does the rank needed also double per 6 dB, or does it grow faster? The singular-value decay curve tells you before you measure.

> 🇪🇸 "Rango 20" no es una decisión defendible; "el menor rango que alcanza 25 dB" sí lo es.
>
> Da la vuelta al requisito: elige un objetivo de calidad, encuentra el rango más barato que lo cumple y reporta lo que ese rango cuesta almacenar.
>
> **Predice antes de ejecutar:** cada 6 dB extra es aproximadamente reducir el error a la mitad. ¿El rango necesario también se duplica cada 6 dB o crece más rápido? La curva de decaimiento de los valores singulares lo dice antes de medir.

In [ ]:
# TODO 4 / TAREA 4
#
# EN:
# 1. Write a function needed_rank(image, target_db) that returns the smallest
#    k with psnr(image, rank-k truncation) >= target_db.
#    Do it WITHOUT rebuilding the truncation for every k: Eckart-Young gives
#    the squared error directly as sum(S[k:] ** 2), so
#       mse(k) = sum(S[k:] ** 2) / (m * n)
#    and PSNR follows from that. One SVD, no loop over reconstructions.
# 2. Run it on the astronaut image for 20, 25, 30 and 35 dB.
# 3. For each, print the rank, the numbers stored k*(m + n + 1), and the
#    percentage of mn that represents.
# 4. Verify one of your answers by actually building the truncation and
#    calling psnr on it.
# 5. Plot required rank against target dB. Is the growth linear?
#
# ES:
# 1. Escribe needed_rank(image, target_db) que devuelva el menor k con
#    psnr >= target_db, SIN reconstruir para cada k: por Eckart-Young el
#    error cuadrático es sum(S[k:] ** 2), así que mse(k) = eso / (m * n).
# 2. Ejecútala sobre la imagen astronaut para 20, 25, 30 y 35 dB.
# 3. Para cada una, imprime el rango, los números almacenados y su
#    porcentaje frente a mn.
# 4. Verifica una respuesta construyendo la truncación y llamando a psnr.
# 5. Grafica el rango necesario frente al objetivo en dB. ¿Es lineal?

In [ ]:
#@title Solution / Solución — try it yourself first / inténtalo primero { display-mode: 'form' }

def needed_rank(A, target_db, peak=1.0):
    """Smallest k reaching target_db, read straight off the singular values."""
    m, n = A.shape
    S = np.linalg.svd(A, compute_uv=False)
    # Eckart-Young: ||A - A_k||_F^2 = sum(S[k:] ** 2), so the MSE of the
    # rank-k truncation is that divided by the number of pixels.
    tail = np.concatenate([np.cumsum(S[::-1] ** 2)[::-1], [0.0]])
    mse = tail / (m * n)
    with np.errstate(divide="ignore"):
        db = 10 * np.log10(peak ** 2 / mse)
    return int(np.argmax(db >= target_db))

targets = [20, 25, 30, 35]
m_a, n_a = image.shape
ranks = [needed_rank(image, t) for t in targets]

print("target dB / objetivo   rank / rango   numbers stored / números   % of mn")
print("-" * 72)
for t, k in zip(targets, ranks):
    stored = k * (m_a + n_a + 1)
    print(f"{t:12d}   {k:14d}   {stored:22,}   {100 * stored / (m_a * n_a):6.2f}%")

check_k = ranks[1]
print()
print(f"Verification / Verificación at k = {check_k}:",
      f"{psnr(image, truncate(check_k)):.2f} dB",
      f"(target / objetivo {targets[1]} dB)")
print(f"One rank lower / Un rango menos, k = {check_k - 1}:",
      f"{psnr(image, truncate(check_k - 1)):.2f} dB")

fig, ax = plt.subplots(figsize=(6.5, 3.4))
ax.plot(targets, ranks, "o-")
ax.set_xlabel("Target PSNR (dB) / Objetivo")
ax.set_ylabel("Smallest rank / Rango mínimo")
ax.set_title("Quality is cheap at first and expensive later")
ax.grid(alpha=0.25)
plt.show()

print()
print("EN: the rank needed grows faster than linearly in dB. The singular")
print("    values decay quickly at first and then flatten, so the last few dB")
print("    cost more rank than the first twenty did.")
print("ES: el rango necesario crece más que linealmente en dB. Los valores")
print("    singulares decaen rápido y luego se aplanan, así que los últimos dB")
print("    cuestan más rango que los veinte primeros.")

## 9.5 Rank that moves a real metric / Rango que mueve una métrica real

<div style="height:3px;border-radius:2px;margin:1.6em 0 1.9em;background:linear-gradient(90deg,#c2410c,rgba(194,65,12,0))"></div>

A reconstruction metric is only a proxy. Check the downstream task that actually matters.

Fit the PCA basis on training rows. Project held-out digits into it. Compare classification accuracy across `k`.

Find the smallest `k` within one percentage point of full-rank accuracy.

<div style="border-left:4px solid rgba(130,130,150,.5);background:rgba(130,130,150,.09);border-radius:0 8px 8px 0;padding:15px 19px;margin:1.8em 0 2.2em;font:400 14.5px/1.8 ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"><div style="font:700 10.5px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;opacity:.62;margin-bottom:10px">🇪🇸 ESPAÑOL</div>Ajusta PCA solo con entrenamiento y proyecta los dígitos reservados. Busca el menor `k` a un punto porcentual del acierto con rango completo.</div>

In [ ]:
# A fixed 70/30 split of the real digits, and the PCA basis of the training
# half only — fitting the basis on the test rows would leak. The split gets its
# own generator so that re-running any cell above cannot change it.
split = np.random.default_rng(12).permutation(len(D))
cut = int(0.7 * len(D))
train_idx, test_idx = split[:cut], split[cut:]

D_train, D_test = D[train_idx], D[test_idx]
y_train, y_test = digit_labels[train_idx], digit_labels[test_idx]

pixel_mean = D_train.mean(axis=0)
U_d, S_d, Vt_d = np.linalg.svd(D_train - pixel_mean, full_matrices=False)

targets_train = np.eye(10)[y_train]

def accuracy_at_rank(k):
    """Least-squares classifier on the top-k principal directions."""
    Z_train = np.column_stack([
        np.ones(len(D_train)), (D_train - pixel_mean) @ Vt_d[:k].T])
    Z_test = np.column_stack([
        np.ones(len(D_test)), (D_test - pixel_mean) @ Vt_d[:k].T])
    W, *_ = np.linalg.lstsq(Z_train, targets_train, rcond=None)
    return np.mean((Z_test @ W).argmax(axis=1) == y_test)

rank_grid = np.arange(1, D.shape[1] + 1)
accuracy_curve = np.array([accuracy_at_rank(int(k)) for k in rank_grid])

full_rank_accuracy = accuracy_curve[-1]
best_k = int(rank_grid[np.argmax(accuracy_curve)])
within_one_point = int(rank_grid[
    np.argmax(accuracy_curve >= full_rank_accuracy - 0.01)])

print("Train / Entrenamiento:", D_train.shape,
      "  Test / Prueba:", D_test.shape)
print()
print("Full rank (k = 64) accuracy / Exactitud a rango completo:",
      f"{full_rank_accuracy:.4f}")
print("Best accuracy / Mejor exactitud:",
      f"{accuracy_curve.max():.4f}", f"at k = {best_k}")
print("First k within 1 point of full rank / Primer k a 1 punto:",
      within_one_point,
      f"({100 * within_one_point / D.shape[1]:.0f}% of the features)")

### Interactive downstream-task explorer / Explorador interactivo de la tarea final

Slide the rank and watch the classifier's accuracy against what the same rank does to a single digit image. Low rank blurs the digit into a smear long before accuracy collapses — the classifier does not need the digit to *look* right, only to be *separable*.

That gap is the reason reconstruction error is a bad stopping rule. It measures how well you kept the picture; the metric you ship measures whether the decision survived.

> 🇪🇸 Desliza el rango y compara la exactitud del clasificador con lo que ese mismo rango le hace a un dígito concreto. Un rango bajo convierte el dígito en una mancha mucho antes de que la exactitud se desplome. El clasificador no necesita que el dígito *se vea* bien, solo que sea *separable*.
>
> Esa brecha es la razón por la que el error de reconstrucción es una mala regla de parada. Mide lo bien que conservaste la imagen, no si sobrevivió la decisión.

In [ ]:
#@title Task accuracy / Acierto de la tarea — run me / ejecútame { display-mode: 'form' }

downstream_rank = widgets.IntSlider(
    value=16,
    min=1,
    max=64,
    step=1,
    description="Rank k / Rango k:",
    continuous_update=False,
    style={"description_width": "130px"},
)

digit_pick = widgets.IntSlider(
    value=0,
    min=0,
    max=19,
    step=1,
    description="Test digit / Dígito:",
    continuous_update=False,
    style={"description_width": "130px"},
)

def explore_downstream(k, which):
    accuracy = accuracy_curve[k - 1]
    reconstructed = (pixel_mean
                     + ((D_test - pixel_mean) @ Vt_d[:k].T) @ Vt_d[:k])
    error = relative_error(D_test, reconstructed)

    print("Rank / Rango:", k, "of / de 64")
    print("Held-out accuracy / Exactitud reservada:", f"{accuracy:.4f}")
    print("Full-rank accuracy / A rango completo:",
          f"{full_rank_accuracy:.4f}",
          f"({accuracy - full_rank_accuracy:+.4f})")
    print("Reconstruction error / Error de reconstrucción:", f"{error:.4f}")

    fig, axes = plt.subplots(1, 3, figsize=(11.0, 3.4))
    axes[0].plot(rank_grid, accuracy_curve, "-", linewidth=1.5)
    axes[0].axvline(k, color="0.6", linewidth=1)
    axes[0].axhline(full_rank_accuracy, color="0.75", linestyle=":",
                    linewidth=1)
    axes[0].set_xlabel("Rank k / Rango k")
    axes[0].set_ylabel("Held-out accuracy / Exactitud")
    axes[0].set_title("Accuracy against rank / Exactitud frente al rango")
    axes[0].grid(alpha=0.25)

    axes[1].imshow(D_test[which].reshape(8, 8), cmap="gray_r")
    axes[1].set_title(f"Original — label / etiqueta {y_test[which]}")
    axes[2].imshow(reconstructed[which].reshape(8, 8), cmap="gray_r")
    axes[2].set_title(f"Rank {k} / Rango {k}")
    for ax in axes[1:]:
        ax.axis("off")
    plt.tight_layout()
    plt.show()

downstream_output = widgets.interactive_output(
    explore_downstream,
    {"k": downstream_rank, "which": digit_pick},
)

display(widgets.VBox([downstream_rank, digit_pick, downstream_output]))

## 9.6 NMF — parts you can name / NMF — partes que puedes nombrar

<div style="height:3px;border-radius:2px;margin:1.6em 0 1.9em;background:linear-gradient(90deg,#c2410c,rgba(194,65,12,0))"></div>

NMF trades optimal error for non-negative, additive parts that people can inspect.

Compare NMF and SVD at rank 16: components, error, and near-zero entries.

NMF cannot beat the optimal rank-16 SVD error. Nonnegativity does not guarantee sparsity or meaningful parts.

<div style="border-left:4px solid rgba(130,130,150,.5);background:rgba(130,130,150,.09);border-radius:0 8px 8px 0;padding:15px 19px;margin:1.8em 0 2.2em;font:400 14.5px/1.8 ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"><div style="font:700 10.5px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;opacity:.62;margin-bottom:10px">🇪🇸 ESPAÑOL</div>Compara componentes, error y valores casi nulos a rango 16. NMF no puede mejorar el error óptimo de SVD. La no negatividad no garantiza dispersión ni partes significativas.</div>

In [ ]:
k_parts = 16

nmf = NMF(n_components=k_parts, init="nndsvda", max_iter=2000,
          tol=1e-5, random_state=0)
W = nmf.fit_transform(D)
H = nmf.components_

U_D, S_D, Vt_D = np.linalg.svd(D, full_matrices=False)
D_svd = (U_D[:, :k_parts] * S_D[:k_parts]) @ Vt_D[:k_parts]

near_zero = lambda M: np.mean(np.abs(M) < 0.01 * np.abs(M).max())

print(f"Rank / Rango {k_parts} on / sobre {D.shape}")
print()
print("Relative error / Error relativo:")
print("   SVD (optimal / óptima):", f"{relative_error(D, D_svd):.4f}")
print("   NMF                   :", f"{relative_error(D, W @ H):.4f}")
print()
print("Component entries that are effectively zero / Entradas casi nulas:")
print("   SVD components / componentes:", f"{100 * near_zero(Vt_D[:k_parts]):.1f}%")
print("   NMF components / componentes:", f"{100 * near_zero(H):.1f}%")
print("   Most negative SVD entry / Entrada más negativa:",
      f"{Vt_D[:k_parts].min():.3f}")
print("   Most negative NMF entry / Entrada más negativa:", f"{H.min():.3f}")

fig, axes = plt.subplots(2, k_parts // 2, figsize=(12.0, 3.6))
for i, ax in enumerate(axes.ravel()):
    ax.imshow(H[i].reshape(8, 8), cmap="gray_r")
    ax.axis("off")
fig.suptitle("NMF components — each one is a stroke, not a correction "
             "/ cada componente es un trazo, no una corrección")
plt.tight_layout()
plt.show()

fig, axes = plt.subplots(2, k_parts // 2, figsize=(12.0, 3.6))
for i, ax in enumerate(axes.ravel()):
    ax.imshow(Vt_D[i].reshape(8, 8), cmap="coolwarm",
              vmin=-np.abs(Vt_D[i]).max(), vmax=np.abs(Vt_D[i]).max())
    ax.axis("off")
fig.suptitle("SVD components — red adds, blue subtracts "
             "/ el rojo suma, el azul resta")
plt.tight_layout()
plt.show()

print()
print("EN: the SVD wins on error by a few points and loses on readability")
print("    completely. Which one you want depends on whether a human has to")
print("    explain the components afterwards.")
print("ES: la SVD gana en error por unos puntos y pierde por completo en")
print("    legibilidad. Cuál quieres depende de si después alguien tiene que")
print("    explicar las componentes.")

## 9.7 Sizing your own problem / Dimensiona tu propio problema

<div style="height:3px;border-radius:2px;margin:1.6em 0 1.9em;background:linear-gradient(90deg,#c2410c,rgba(194,65,12,0))"></div>

Start with shape, budget, and goal. The method follows from those three.

Set `m`, `n`, and `k`. Compare work, bytes, and estimated time.

Time is calibrated using one local QR run. Other methods may achieve different processing rates. Treat it as a rough estimate.

<div style="border-left:4px solid rgba(130,130,150,.5);background:rgba(130,130,150,.09);border-radius:0 8px 8px 0;padding:15px 19px;margin:1.8em 0 2.2em;font:400 14.5px/1.8 ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"><div style="font:700 10.5px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;opacity:.62;margin-bottom:10px">🇪🇸 ESPAÑOL</div>Fija `m`, `n` y `k`. Compara operaciones, bytes y tiempo estimado. La calibración usa una QR local; otros métodos pueden rendir distinto. Es una aproximación.</div>

In [ ]:
#@title Budget explorer / Explorador de presupuesto — run me / ejecútame { display-mode: 'form' }

# Calibrate on one QR of known flop count, on this machine, right now.
calibration_n = 512
calibration_matrix = np.random.default_rng(2).standard_normal(
    (calibration_n, calibration_n))
calibration_time = best_time(lambda: np.linalg.qr(calibration_matrix))
calibration_flops = FLOPS["QR"](calibration_n, calibration_n, 0)
flop_rate = calibration_flops / calibration_time

print(f"Calibration / Calibración: QR {calibration_n}x{calibration_n} in "
      f"{calibration_time * 1e3:.1f} ms")
print(f"Effective rate / Tasa efectiva: {flop_rate / 1e9:.1f} GFLOP/s")
print()

rows_slider = widgets.IntSlider(
    value=10_000, min=100, max=200_000, step=100,
    description="Rows m / Filas m:", continuous_update=False,
    style={"description_width": "150px"}, readout_format=",d")
cols_slider = widgets.IntSlider(
    value=500, min=10, max=5_000, step=10,
    description="Columns n / Columnas n:", continuous_update=False,
    style={"description_width": "150px"}, readout_format=",d")
target_rank = widgets.IntSlider(
    value=20, min=1, max=500, step=1,
    description="Target rank k / Rango k:", continuous_update=False,
    style={"description_width": "150px"})

def human_time(seconds):
    if seconds < 1e-3:
        return f"{seconds * 1e6:.0f} us"
    if seconds < 1:
        return f"{seconds * 1e3:.0f} ms"
    if seconds < 90:
        return f"{seconds:.1f} s"
    if seconds < 5400:
        return f"{seconds / 60:.1f} min"
    return f"{seconds / 3600:.1f} h"

def human_bytes(n_bytes):
    for unit in ("B", "KB", "MB", "GB", "TB"):
        if n_bytes < 1024 or unit == "TB":
            return f"{n_bytes:,.1f} {unit}"
        n_bytes /= 1024

def budget(m, n, k):
    if n > m:
        m, n = n, m
        print("EN: m and n swapped so that m >= n, as the formulas assume.")
        print("ES: se intercambian m y n para que m >= n, como suponen las")
        print("    fórmulas.")
        print()

    k = min(k, n)
    raw_bytes = m * n * 8

    # Numbers each method has to keep, in float64.
    STORED = {
        "Cholesky": n * (n + 1) / 2,
        "LU": n * n,
        "QR": m * n + n * (n + 1) / 2,
        "Eigendecomposition": n * n + n,
        "Thin SVD": m * n + n + n * n,
        "Randomized SVD": k * (m + n + 1),
    }

    # Cholesky, LU and the eigendecomposition are only defined for a square
    # matrix, so their formulas must not be quoted for a rectangular one.
    SQUARE_ONLY = {"Cholesky", "LU", "Eigendecomposition"}
    square = m == n

    print(f"A is {m:,} x {n:,} float64 = {human_bytes(raw_bytes)},"
          f" target rank k = {k}")
    if not square:
        print("A is rectangular / A es rectangular — Cholesky, LU and the")
        print("eigendecomposition do not apply / no se aplican.")
    print()
    print("method / método".ljust(20)
          + "flops".rjust(11) + "est. time".rjust(11)
          + "factors".rjust(13) + "  vs A")
    print("-" * 68)
    for name, formula in FLOPS.items():
        if name in SQUARE_ONLY and not square:
            print(name.ljust(20)
                  + "n/a — needs a square matrix / requiere una cuadrada".rjust(
                      41))
            continue
        flops = formula(m, n, k)
        stored_bytes = STORED[name] * 8
        print(name.ljust(20)
              + f"{flops:10.2e}"
              + human_time(flops / flop_rate).rjust(11)
              + human_bytes(stored_bytes).rjust(13)
              + f"  {stored_bytes / raw_bytes:6.2f}x")

    print()
    full = FLOPS["Thin SVD"](m, n, k)
    randomized = FLOPS["Randomized SVD"](m, n, k)
    print(f"Full SVD / randomized SVD at k = {k}: "
          f"{full / randomized:.1f}x more work / más trabajo")
    print("EN: that ratio is the whole argument for not computing what you")
    print("    intend to throw away.")
    print("ES: esa razón es todo el argumento para no calcular lo que vas a")
    print("    tirar.")

budget_output = widgets.interactive_output(
    budget,
    {"m": rows_slider, "n": cols_slider, "k": target_rank},
)

display(widgets.VBox([rows_slider, cols_slider, target_rank, budget_output]))

## Quick decision challenge / Reto rápido de decisión

Six situations, one factorization each. Pick before you read the answer.

> 🇪🇸 Seis situaciones, una factorización para cada una. Elige antes de leer la respuesta.

In [ ]:
#@title Decision challenge / Reto de decisión — run me / ejecútame { display-mode: 'form' }

scenario = widgets.Dropdown(
    options=[
        ("A 50,000 x 300 design matrix, fit once / una vez", "tall"),
        ("A 4,000 x 4,000 covariance matrix, 500 right-hand sides",
         "spd_many"),
        ("A 200,000 x 50,000 sparse ratings matrix, top 30 components",
         "sparse_lowrank"),
        ("A 3,000 x 3,000 gene-expression matrix, components must be "
         "explainable", "nonneg"),
        ("A Markov transition matrix — what is the steady state?", "markov"),
        ("A 40,000 x 8,000 dense matrix, top 100 components", "dense_lowrank"),
    ],
    value="tall",
    description="Situation / Situación:",
    style={"description_width": "150px"},
    layout=widgets.Layout(width="720px"),
)

ANSWERS = {
    "tall": (
        "QR (np.linalg.lstsq)",
        "Tall and fit once: this is least squares. Never form X^T X — QR "
        "solves it at kappa(X) instead of kappa(X)^2.",
        "Alta y ajustada una vez: esto es mínimos cuadrados. Nunca formes "
        "X^T X — QR lo resuelve con kappa(X) en vez de kappa(X)^2.",
    ),
    "spd_many": (
        "Cholesky (cho_factor once, then cho_solve 500 times)",
        "A covariance matrix is symmetric positive definite, so Cholesky is "
        "half the flops of LU. 500 right-hand sides make the factor-once "
        "saving O(n^3 + m n^2) against O(m n^3).",
        "Una matriz de covarianza es simétrica definida positiva, así que "
        "Cholesky cuesta la mitad que LU. Con 500 lados derechos, factorizar "
        "una vez es O(n^3 + m n^2) frente a O(m n^3).",
    ),
    "sparse_lowrank": (
        "Lanczos / truncated SVD (scipy.sparse.linalg.svds)",
        "Sparse and only 30 components wanted. A dense SVD would first "
        "densify a 10-billion-entry matrix to compute 30 vectors. Lanczos "
        "costs O(k * nnz(A)) and never densifies.",
        "Dispersa y solo se quieren 30 componentes. Una SVD densa primero "
        "densificaría una matriz de 10 mil millones de entradas para calcular "
        "30 vectores. Lanczos cuesta O(k * nnz(A)) y nunca densifica.",
    ),
    "nonneg": (
        "NMF",
        "Expression counts are nonnegative and the components have to be "
        "readable. This is the one case where giving up Eckart-Young "
        "optimality is the right call — you are buying interpretability with "
        "error.",
        "Los conteos de expresión son no negativos y las componentes deben ser "
        "legibles. Es el único caso donde renunciar a la optimalidad de "
        "Eckart-Young es correcto: compras interpretabilidad con error.",
    ),
    "markov": (
        "Eigendecomposition (or power iteration)",
        "Steady state is the eigenvector for eigenvalue 1 — the question "
        "'what does repeated application converge to?' from section 08. If "
        "you only need the top one, power iteration beats a full "
        "eigendecomposition.",
        "El estado estacionario es el autovector del autovalor 1: la pregunta "
        "'¿a qué converge la aplicación repetida?' de la sección 08. Si solo "
        "necesitas el primero, la iteración de potencias gana a una "
        "descomposición completa.",
    ),
    "dense_lowrank": (
        "Randomized SVD (sklearn.utils.extmath.randomized_svd)",
        "Dense, so Lanczos has no sparsity to exploit, but 100 components out "
        "of 8,000 means a full SVD does eighty times the necessary work. "
        "Randomized SVD costs O(mnk).",
        "Densa, así que Lanczos no tiene dispersión que aprovechar, pero 100 "
        "componentes de 8.000 significa que una SVD completa hace ochenta "
        "veces el trabajo necesario. La SVD aleatorizada cuesta O(mnk).",
    ),
}

def answer(key):
    pick, en, es = ANSWERS[key]
    print("Use / Usa:", pick)
    print()
    print("EN:", en)
    print()
    print("ES:", es)

challenge_output = widgets.interactive_output(answer, {"key": scenario})

display(widgets.VBox([scenario, challenge_output]))

## What just happened / Qué acaba de pasar

<div style="height:3px;border-radius:2px;margin:1.6em 0 1.9em;background:linear-gradient(90deg,#c2410c,rgba(194,65,12,0))"></div>

Choose by the problem: solve, fit, compress, interpret, or reduce. Then check the assumptions.

<div style="border-left:4px solid rgba(130,130,150,.5);background:rgba(130,130,150,.09);border-radius:0 8px 8px 0;padding:15px 19px;margin:1.8em 0 2.2em;font:400 14.5px/1.8 ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"><div style="font:700 10.5px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;opacity:.62;margin-bottom:10px">🇪🇸 ESPAÑOL</div>Elige según el problema: resolver, ajustar, comprimir, interpretar o reducir. Después revisa los supuestos.</div>

## Where this goes next / Adónde sigue esto

<div style="height:3px;border-radius:2px;margin:1.6em 0 1.9em;background:linear-gradient(90deg,#c2410c,rgba(194,65,12,0))"></div>

Choose a topic to revisit. Posts are in English.

<details>
<summary>Further reading / Lecturas adicionales (inglés)</summary>

- [Why so many matrix factorizations?](https://project-delphi.github.io/ml-blog/posts/why-so-many-matrix-factorizations/)
- [Factorizations as optimization](https://project-delphi.github.io/ml-blog/posts/matrix-factorizations/)
- [Rotate, stretch, rotate again](https://project-delphi.github.io/ml-blog/posts/svd-rotate-stretch-rotate/)
- [The directions a matrix refuses to turn](https://project-delphi.github.io/ml-blog/posts/eigendecomposition/)
- [Can you invert a recursive function?](https://project-delphi.github.io/ml-blog/posts/recursive-inversion/)

</details>

<div style="border-left:4px solid rgba(130,130,150,.5);background:rgba(130,130,150,.09);border-radius:0 8px 8px 0;padding:15px 19px;margin:1.8em 0 2.2em;font:400 14.5px/1.8 ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"><div style="font:700 10.5px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;opacity:.62;margin-bottom:10px">🇪🇸 ESPAÑOL</div>Elige el tema que quieras repasar. Las lecturas adicionales están en inglés.</div>

<div style="height:3px;border-radius:2px;margin:1.4em 0 1.6em;background:linear-gradient(90deg,#c2410c,rgba(194,65,12,0))"></div>

## Done with this section / Fin de esta sección

Next / Siguiente: **10 · Tucker decomposition on real data / Descomposición de Tucker con datos reales** — [open in Colab](https://colab.research.google.com/github/project-delphi/tensors-workshop/blob/main/notebooks/10-tucker-decomposition.ipynb).

[← Workshop site / Sitio del taller](https://project-delphi.github.io/tensors-workshop/) · [All notebooks / Todos los notebooks](https://project-delphi.github.io/tensors-workshop/notebooks.html) · [Handbook / Manual](https://project-delphi.github.io/tensors-workshop/tensors_workshop_plan_with_quizzes.html)